# PKG Attrition v13 · Model 2 — change over size, and a tree model

**Payment Knowledge Graph (PKG) · Treasury Management · Data Science**

Model 1 mostly learned *which kind of client* leaves (small, low-activity) rather than *what changes before they
leave*: contributions were flat from 12 months out, levels dominated, missing-value flags took a quarter of the
weight, and near-duplicate features took opposite signs. Model 2 changes five things:

| | Change | Why |
|---|---|---|
| 1 | Train and score only **money-still-here** client-months | the list only calls those; model 1 trained mostly on tiny and near-empty clients |
| 2 | Payment levels become **gaps to size peers** (same month, same normal-balance decile); three size anchors stay | a level says how big a client is; the gap says whether it behaves unlike clients its size |
| 3 | Near-duplicates dropped (|r| > 0.95), one missing-share per feature set, ridge penalty chosen on validation | readable signs and reason codes |
| 4 | A **gradient-boosted tree model** (XGBoost → LightGBM → sklearn, whichever is installed) beside the simple model, same folds, same rows | how much a stronger model buys, judged on dollars reached |
| 5 | **The largest departures are checked** from the data: did the money come back, move to another PNC customer in the relationship, or keep transacting here? | the dollars rest on a few very large clients |

Event-time signal timing is rebuilt on **change features only, against stayers of the same size band**, and
drains already in progress (within-month range, month-end vs average, last-month change) are reported
separately as late warnings. Sources are unchanged: the deposit table and `pkg_cust_metrics_monthly`; no table is
written. Expect 1–3 hours, mostly the tree folds.

### Pre-registered predictions — scored in §12

| # | Prediction | Confidence |
|---|---|---|
| P1 | Tree model pooled AUC (money-still-here rows) ≥ simple model's + 0.02 | 60% |
| P2 | Tree model reaches more capped dollars at 1,000 calls, α = 0.5, 95% paired interval above zero | 45% |
| P3 | The model-view gap between leavers and stayers at 6 months out is ≤ 60% of the gap at 1 month out (change-driven now) | 55% |
| P4 | ≥ 3 payment change features fire at or before 6 months out against size-matched stayers | 45% |
| P5 | Of the largest departures checked, ≥ 3 are not clean departures | 50% |
| P6 | Removing the 5 largest departing clients cuts dollars reached (1,000 calls, α = 0.5) by ≥ 30% | 50% |
| P7 | Tree model: best month's precision ≤ 2× worst month's (1,000 calls, α = 0.5) | 60% |

In [ ]:
# =====================================================================
# 0 · CONFIG
# =====================================================================
DEP_TABLE  = "dsihd01p_dsi.lap_dsi_universe_optimized"          # daily: account × leg × business day
MET_TABLE  = "bdahd01p_dlcdi1_cdi_tm.pkg_cust_metrics_monthly"
DICT_TABLE = "bdahd01p_dlcdi1_cdi_tm.pkg_cust_metrics_dictionary"
OUT_DIR    = "../metrics/attrition/v13/m2"
DATE_START, DATE_END = "2024-01-01", "2026-08-31"                # nothing before 2024 is trusted
SHUFFLE_PARTITIONS = 800

# ── status codes: two vocabularies in one column (settled in v2) ─────
STATUS_CLASS = {"01": "open", "99": "open", "09": "open", "O": "open",
                "03": "inactive", "05": "dormant", "06": "escheat", "12": "closing",
                "07": "closed", "08": "closed", "C": "closed"}
LIVE_CLASSES = {"open", "inactive", "dormant", "escheat", "closing"}
PRICING = ["earnings_credit_rate", "curr_int_rate"]              # used if present

# ── definitions ──────────────────────────────────────────────────────
H          = 6          # horizon: does the money leave within the next 6 months?
CONFIRM_M  = 2          # a full exit is confirmed after 2 observed non-live months
NORM_WIN, NORM_MIN_OBS = 12, 3    # normal balance: median of up to 12 prior live months, needs 3
MIN_HIST   = 3          # months observed before a client-month can be scored
MSH_FRAC, MSH_FLOOR = 0.50, 10_000     # money still here: ≥ half of normal, normal ≥ $10k
BAL_EXIT_FRAC, BAL_EXIT_HOLD = 0.05, 3 # balance exit: < 5% of normal for 3 months, accounts open
CHURN_WARM = 6          # months of payment history before the churn-state metrics are used

# ── training ─────────────────────────────────────────────────────────
NEG_RATE   = 0.15       # share of non-leaving client-months kept for training (weight 1/NEG_RATE)
SEED       = 13
L2_GRID    = [1.0, 10.0, 100.0, 1000.0]   # ridge penalty (summed log-likelihood), chosen on the first fold's validation
CLIP_Q     = (0.005, 0.995)
CORR_MAX   = 0.95       # drop the later of two features correlated above this
TEST_FIRST = "2025-04"  # first test month; the last is the last month with a fully observed outcome
ENGINE     = "auto"     # "auto" | "xgboost" | "lightgbm" | "sklearn" | "off"
GBM_GRID   = [4, 6, 8]  # tree depth tried on the first fold's validation
GBM_ETA, GBM_MAX_TREES, GBM_EARLY = 0.05, 3000, 100

# ── queue and value ──────────────────────────────────────────────────
CALLS      = [100, 250, 500, 1000, 2000]   # calls per month
ALPHAS     = [0.0, 0.5, 1.0]               # score = p × normal balance^α
SAVE_RATES = [0.01, 0.05, 0.10, 0.20]      # illustrative; no call cost anywhere
K_MAIN, A_MAIN = 1000, 0.5
BANDS      = [(0, 1e5, "< $100k"), (1e5, 1e6, "$100k–1m"), (1e6, 1e7, "$1m–10m"), (1e7, float("inf"), "$10m+")]

# ── signal timing and examples ───────────────────────────────────────
N_TIMING   = 30         # features traced month by month before exit
LIFT_BAR   = 1.5        # a signal "fires" when leavers cross the stayers' 10th/90th percentile ≥ 1.5× as often
EVENT_PRE  = 12
MAX_EVENT_CUST = 15_000 # per group, for the event-time pulls
N_SHAP_CUST = 5_000     # per group, for the model view
LATE_FEATS = ["dep_range", "dep_eom_vs_avg", "dep_d1"]   # drain already in progress: late warnings
N_AUDIT    = 15         # largest departures checked, from the whole window and from the test months each
TOP_EXCL   = [1, 5]     # headline dollars also shown without the largest 1 and 5 departing clients

In [ ]:
# =====================================================================
# 1 · SETUP — shims, session, helpers
# =====================================================================
import numpy as np, warnings
for _a, _t in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _a): setattr(np, _a, _t)
if "bool" not in np.__dict__: np.bool = np.bool_
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"): pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):    pd.Series.iteritems = pd.Series.items
for _c in (ResourceWarning, DeprecationWarning, FutureWarning, RuntimeWarning):
    warnings.filterwarnings("ignore", category=_c)

import os, re, time, json, base64, hashlib, pathlib, html as _html
from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")            # features are cast to float before collecting
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
DISK = StorageLevel.DISK_ONLY
OUT = pathlib.Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()
R = {}          # every number the report uses
CHARTS = {}     # chart name → file
CHECKS = []

# ── one consolidated output per cell ─────────────────────────────────
def _fmt_col(s, kind):
    if kind == "pct":   return s.map(lambda v: "—" if pd.isna(v) else f"{v:.1%}")
    if kind == "money": return s.map(lambda v: "—" if pd.isna(v) else f"${v:,.0f}")
    if pd.api.types.is_integer_dtype(s): return s.map(lambda v: f"{v:,}")
    if pd.api.types.is_float_dtype(s):   return s.map(lambda v: "—" if pd.isna(v) else f"{v:,.3f}")
    return s.map(lambda v: "—" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))

def fmt_df(df, pct=(), money=()):
    d = df.copy()
    for c in d.columns:
        d[c] = _fmt_col(d[c], "pct" if c in pct else "money" if c in money else None)
    return d

class Block:
    """Collects every table and note of a cell and displays them once, as one image-friendly output."""
    def __init__(self, title):
        self.h = [f"<div style='font-family:Arial,sans-serif;font-size:12px'><h3 style='margin:2px 0 6px'>{title}</h3>"]
    def table(self, df, title, pct=(), money=(), save=None, max_rows=80):
        if save: df.to_csv(OUT / f"{save}.csv", index=False)
        d = fmt_df(df.head(max_rows), pct, money)
        self.h.append(f"<p style='margin:8px 0 2px'><b>{title}</b> <span style='color:#777'>[{len(df):,} rows]</span></p>" +
                      d.to_html(index=False, border=0, classes="t").replace(
                          "<table", "<table style='border-collapse:collapse;font-size:11px' cellpadding='3'"))
        return self
    def kv(self, pairs, title):
        return self.table(pd.DataFrame([{"item": k, "value": v if isinstance(v, str) else
                          f"{v:,}" if isinstance(v, (int, np.integer)) else f"{v:,.4f}" if isinstance(v, float) else str(v)}
                                        for k, v in pairs]), title)
    def note(self, s):
        self.h.append(f"<p style='margin:4px 0;color:#333'>{s}</p>"); return self
    def check(self, item, status, detail):
        CHECKS.append({"item": item, "status": status, "detail": detail})
        col = {"READY": "#2e7d32", "FIX": "#e65100", "BLOCKER": "#b71c1c", "INFO": "#555"}[status]
        return self.note(f"<span style='color:{col};font-weight:bold'>[{status}]</span> {item}: {detail}")
    def show(self, t0=None):
        if t0: self.note(f"<span style='color:#999'>wall {time.time() - t0:,.0f}s</span>")
        display(HTML("".join(self.h) + "</div>"))

def savefig(fig, name):
    p = OUT / f"{name}.png"; fig.savefig(p, dpi=120, bbox_inches="tight"); plt.close(fig); CHARTS[name] = p

def topd(df):
    for c, t in df.dtypes:
        if t.startswith("decimal"): df = df.withColumn(c, F.col(c).cast("double"))
    return df.toPandas()

def nz(c):
    s = F.trim(F.col(c).cast("string")); return F.when(s == "", F.lit(None)).otherwise(s)

# ── months: offset t from the first month only ───────────────────────
def ym_idx(ym): y, m = map(int, ym[:7].split("-")); return y * 12 + m - 1
M0 = ym_idx(DATE_START)
def t_of(ym): return ym_idx(ym) - M0
def ym_of(t): i = M0 + int(t); return f"{i // 12:04d}-{i % 12 + 1:02d}"
def t_col(c): return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - 1 - F.lit(M0)).cast("int")
assert t_of(DATE_START) == 0 and ym_of(0) == DATE_START[:7]

def cid(x):
    """Display id: one-way hash. Full ids are only ever written to *_FULL_IDS.csv."""
    return hashlib.sha1(str(x).encode()).hexdigest()[:10]

def auc(y, s):
    """Rank AUC with ties averaged; y in {0,1}."""
    y = np.asarray(y, float); s = np.asarray(s, float); ok = ~np.isnan(s); y, s = y[ok], s[ok]
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

# ── tree engine ──────────────────────────────────────────────────────
def _engine():
    order = ["xgboost", "lightgbm", "sklearn"] if ENGINE == "auto" else ([] if ENGINE == "off" else [ENGINE])
    for e in order:
        try:
            importlib.import_module(e if e != "sklearn" else "sklearn.ensemble"); return e
        except Exception: pass
    return None
import importlib
GBM = _engine()
print(f"Spark {spark.version} | window {DATE_START} … {DATE_END} | tree engine: {GBM or 'none (simple model only)'} "
      f"| cpus {os.cpu_count()} | outputs → {OUT.resolve()}")

## §2 Deposit data → monthly customer panel, targets, deposit features

Legs are summed per account-day; balances are rolled to customer-day, then month (average, minimum, maximum,
month-end). Every customer gets a complete month grid from its first month, so a month with no rows reads as
non-live with zero balance. All windows look backwards only.

**Target** (defined and argued in the report, §4): *the money leaves within the next 6 months*, the earlier of
a **full exit** (every account non-live, confirmed by 2 observed non-live months) and a **balance exit** (balance
below 5% of normal for 3 months while accounts stay open).

In [ ]:
# =====================================================================
# 2 · DEPOSIT PANEL, TARGETS, DEPOSIT FEATURES                  [OUTPUT BLOCK 2]
# =====================================================================
t0 = time.time(); B = Block("§2 · Deposit data → monthly customer panel")
raw = spark.table(DEP_TABLE)
REQ = ["acct_full_acct_id", "sub_product_cd", "edw_tda_load_dt", "balance", "acct_status", "deposit_family",
       "opened_dt", "closed_dt", "cust_pwr_id", "rltn_pwr_id"]
miss = [c for c in REQ if c not in raw.columns]
if miss: raise RuntimeError(f"deposit table lacks {miss}")
PRC = [c for c in PRICING if c in raw.columns]
TIEB = [c for c in ["bdh_hdfs_load_ts", "cod_hdfs_load_ts"] if c in raw.columns]
SMAP = F.create_map(*[F.lit(x) for kv_ in STATUS_CLASS.items() for x in kv_])
dep = raw.select(*REQ, *PRC, *TIEB)
for c in ["acct_full_acct_id", "sub_product_cd", "cust_pwr_id", "rltn_pwr_id", "acct_status", "deposit_family"]:
    dep = dep.withColumn(c, nz(c))
for c in ["edw_tda_load_dt", "opened_dt", "closed_dt"]: dep = dep.withColumn(c, F.to_date(F.col(c)))
for c in ["balance"] + PRC: dep = dep.withColumn(c, F.col(c).cast("double"))
dep = (dep.filter(F.col("edw_tda_load_dt").between(F.lit(DATE_START), F.lit(DATE_END)))
          .withColumn("leg", F.coalesce(F.col("sub_product_cd"), F.lit("__NA__")))
          .withColumn("ym", F.date_format("edw_tda_load_dt", "yyyy-MM"))
          .withColumn("status_class", F.coalesce(SMAP[F.col("acct_status")], F.lit("UNMAPPED")))
          .withColumn("live_leg", F.col("status_class").isin(*LIVE_CLASSES).cast("int")))

unm = [r[0] for r in dep.filter("status_class = 'UNMAPPED'").select("acct_status").distinct().limit(20).collect()]
if unm: raise RuntimeError(f"unmapped acct_status codes {unm}: add them to STATUS_CLASS")
n_dup = dep.groupBy("acct_full_acct_id", "leg", "edw_tda_load_dt").count().filter("count > 1").count()
if n_dup:
    _o = [F.col(c).desc_nulls_last() for c in TIEB] + [F.col("balance").desc_nulls_last()]
    dep = dep.withColumn("_rn", F.row_number().over(Window.partitionBy("acct_full_acct_id", "leg", "edw_tda_load_dt")
                                                    .orderBy(*_o))).filter("_rn = 1").drop("_rn")
MC = topd(dep.groupBy("ym").agg(F.countDistinct("edw_tda_load_dt").alias("bdays"),
                                F.countDistinct("cust_pwr_id").alias("customers")).orderBy("ym"))
END_M = MC.loc[MC.bdays >= 15, "ym"].max(); END_T = t_of(END_M)
holes = [ym_of(t) for t in range(END_T + 1) if ym_of(t) not in set(MC.loc[MC.bdays >= 15, "ym"])]
if holes: raise RuntimeError(f"deposit months missing or partial inside the window: {holes}")
dep = dep.withColumn("t", t_col("ym")).filter(F.col("t") <= END_T)

# ── customer-month balances (customer-day first, so daily min / max are real) ──
cday = (dep.filter("cust_pwr_id is not null").groupBy("cust_pwr_id", "edw_tda_load_dt", "t")
           .agg(F.sum("balance").alias("bal_d")))
cbal = cday.groupBy("cust_pwr_id", "t").agg(F.avg("bal_d").alias("bal_avg"), F.min("bal_d").alias("bal_min"),
                                            F.max("bal_d").alias("bal_max"),
                                            F.expr("max_by(bal_d, edw_tda_load_dt)").alias("bal_eom"))
# ── account-month → customer-month counts, product mix, pricing ─────
am = dep.groupBy("acct_full_acct_id", "t", "ym").agg(
    F.max("live_leg").alias("live"),
    F.expr("max_by(cust_pwr_id, edw_tda_load_dt)").alias("cust_pwr_id"),
    F.expr("max_by(rltn_pwr_id, edw_tda_load_dt)").alias("rltn_pwr_id"),
    F.expr("max_by(deposit_family, abs(coalesce(balance, 0)))").alias("deposit_family"),
    (F.sum("balance") / F.countDistinct("edw_tda_load_dt")).alias("acct_bal"),
    F.min("opened_dt").alias("opened_dt"),
    F.max((F.date_format("closed_dt", "yyyy-MM") == F.col("ym")).cast("int")).alias("closed_in_m"),
    *[F.expr(f"max_by({c}, edw_tda_load_dt)").alias(c) for c in PRC])
fam = F.upper(F.coalesce(F.col("deposit_family"), F.lit("")))
pos = F.greatest(F.col("acct_bal"), F.lit(0.0))
fb = lambda cond: F.sum(F.when(cond, pos).otherwise(0.0))
def wavg(c):
    ok = F.col(c).isNotNull() & (F.col("acct_bal") > 0)
    return (F.sum(F.when(ok, F.col(c) * F.col("acct_bal"))) / F.sum(F.when(ok, F.col("acct_bal")))).alias(c)
cacc = am.filter("cust_pwr_id is not null").groupBy("cust_pwr_id", "t").agg(
    F.count("*").alias("n_accts"), F.sum("live").alias("n_live"),
    F.sum((F.date_format("opened_dt", "yyyy-MM") == F.col("ym")).cast("int")).alias("n_opened"),
    F.sum("closed_in_m").alias("n_closed"),
    fb(fam == "IBDDA").alias("fb_ibdda"), fb(fam == "MMDA").alias("fb_mmda"), fb(fam.contains("SWEEP")).alias("fb_sweep"),
    fb(F.lit(True)).alias("fb_all"), F.max("rltn_pwr_id").alias("rltn_pwr_id"), *[wavg(c) for c in PRC])
cm = cbal.join(cacc, ["cust_pwr_id", "t"], "full")

# ── complete month grid ──────────────────────────────────────────────
first = cm.groupBy("cust_pwr_id").agg(F.min("t").alias("first_t"))
grid = first.crossJoin(F.broadcast(spark.range(0, END_T + 1).select(F.col("id").cast("int").alias("t")))) \
            .filter(F.col("t") >= F.col("first_t"))
X = grid.join(cm, ["cust_pwr_id", "t"], "left")
for c in ["n_accts", "n_live", "n_opened", "n_closed"]: X = X.withColumn(c, F.coalesce(F.col(c), F.lit(0)))
for c in ["bal_avg", "bal_min", "bal_max", "bal_eom", "fb_ibdda", "fb_mmda", "fb_sweep", "fb_all"]:
    X = X.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))
wc = Window.partitionBy("cust_pwr_id"); wo = wc.orderBy("t")
rg = lambda a, b: wo.rangeBetween(a, b)
X = (X.withColumn("rltn_pwr_id", F.last("rltn_pwr_id", ignorenulls=True).over(wo.rowsBetween(Window.unboundedPreceding, 0)))
      .withColumn("bal", F.greatest(F.col("bal_avg"), F.lit(0.0)))
      .withColumn("live", (F.col("n_live") > 0).cast("int"))
      .withColumn("hist_t", F.col("t") - F.col("first_t") + 1)
      .withColumn("last_live_t", F.max(F.when(F.col("live") == 1, F.col("t"))).over(wc))
      .withColumn("exit_t", F.when(F.col("last_live_t") < END_T, F.col("last_live_t") + 1).cast("int"))
      .withColumn("a_t", F.when(F.lit(END_T) - F.col("exit_t") + 1 >= CONFIRM_M, F.col("exit_t")))
      .withColumn("_bl", F.when(F.col("live") == 1, F.col("bal")))
      .withColumn("bal_norm", F.expr("percentile_approx(_bl, 0.5)").over(rg(-NORM_WIN, -1)))
      .withColumn("_nn", F.count("_bl").over(rg(-NORM_WIN, -1)))
      .withColumn("bal_norm", F.when(F.col("_nn") >= NORM_MIN_OBS, F.col("bal_norm")))
      .withColumn("msh", F.coalesce(((F.col("live") == 1) & (F.col("bal_norm") >= MSH_FLOOR)
                                     & (F.col("bal") >= MSH_FRAC * F.col("bal_norm"))).cast("int"), F.lit(0)))
      .withColumn("capped", F.least(F.col("bal"), F.col("bal_norm"))))
thr = F.lit(BAL_EXIT_FRAC) * F.col("bal_norm")
bc = (F.col("live") == 1) & (F.col("bal_norm") > 0) & (F.col("bal") < thr) & (F.col("t") + BAL_EXIT_HOLD - 1 <= END_T)
for k in range(1, BAL_EXIT_HOLD): bc = bc & (F.lead("bal", k).over(wo) < thr)
X = (X.withColumn("b_t", F.min(F.when(F.coalesce(bc, F.lit(False)), F.col("t"))).over(wc).cast("int"))
      .withColumn("ab_t", F.least("a_t", "b_t"))
      .withColumn("exit_type", F.when(F.col("ab_t").isNull(), F.lit(None))
                                .when(F.col("a_t") == F.col("ab_t"), F.lit("full exit")).otherwise(F.lit("balance exit"))))

# ── deposit features (backward-looking only) ─────────────────────────
def chg(col, a, b, min_obs=2):
    return F.when(F.count(col).over(rg(a, b)) >= min_obs, F.col(col) - F.avg(col).over(rg(a, b)))
X = X.withColumn("_lb", F.log1p("bal")).withColumn("_nl", F.col("n_live").cast("double"))
DEP = {}
def add(name, expr, group, label):
    global X; X = X.withColumn(name, expr); DEP[name] = (group, label)
add("dep_log_bal", F.col("_lb"), "balance level", "balance, log")
add("dep_bal_vs_norm", F.col("_lb") - F.log1p("bal_norm"), "balance level", "balance vs normal balance (log ratio)")
add("dep_d1", F.col("_lb") - F.lag("_lb", 1).over(wo), "balance change", "balance change vs last month")
add("dep_d3", chg("_lb", -3, -1), "balance change", "balance change vs the last 3 months")
add("dep_d6", chg("_lb", -6, -4), "balance change", "balance change vs 4–6 months ago")
add("dep_d12", chg("_lb", -12, -10), "balance change", "balance change vs 10–12 months ago")
add("dep_drawdown", F.col("_lb") - F.max("_lb").over(rg(-12, -1)), "balance change", "balance vs its 12-month peak")
add("dep_vol6", F.when(F.count("_lb").over(rg(-5, 0)) >= 3, F.stddev("_lb").over(rg(-5, 0))), "balance volatility",
    "volatility of balance over 6 months")
add("dep_range", F.log1p(F.greatest("bal_max", F.lit(0.0))) - F.log1p(F.greatest("bal_min", F.lit(0.0))),
    "balance volatility", "within-month range of daily balance")
add("dep_eom_vs_avg", F.log1p(F.greatest("bal_eom", F.lit(0.0))) - F.col("_lb"), "balance volatility",
    "month-end balance vs monthly average")
add("dep_n_accts", F.log1p("n_accts"), "accounts", "number of accounts")
add("dep_n_live", F.log1p("n_live"), "accounts", "number of live accounts")
add("dep_live_d6", chg("_nl", -6, -4), "accounts", "change in live accounts vs 4–6 months ago")
add("dep_opened_3m", F.sum("n_opened").over(rg(-2, 0)), "accounts", "accounts opened in the last 3 months")
add("dep_closed_3m", F.sum("n_closed").over(rg(-2, 0)), "accounts", "accounts closed in the last 3 months")
for c, lab in [("fb_ibdda", "interest-bearing DDA"), ("fb_mmda", "money market"), ("fb_sweep", "sweep")]:
    add(f"dep_share_{c[3:]}", F.when(F.col("fb_all") > 0, F.col(c) / F.col("fb_all")), "product mix", f"share of balance in {lab}")
add("dep_tenure", F.log1p("hist_t"), "tenure", "months observed (log)")
add("dep_at_start", (F.col("first_t") == 0).cast("int"), "tenure", "client already present when the data starts")
for c in PRC:
    add(f"dep_{c}", F.col(c), "pricing", c.replace("_", " "))
    add(f"dep_{c}_d6", chg(c, -6, -4), "pricing", f"change in {c.replace('_', ' ')} vs 4–6 months ago")
rl = X.filter("rltn_pwr_id is not null").groupBy("rltn_pwr_id", "t").agg(F.sum("bal").alias("rl_bal"),
                                                                         F.countDistinct("cust_pwr_id").alias("rl_n"))
X = X.join(rl, ["rltn_pwr_id", "t"], "left").withColumn(
    "_lo", F.when(F.col("rl_n") > 1, F.log1p(F.greatest(F.col("rl_bal") - F.col("bal"), F.lit(0.0)))))
add("dep_siblings", F.log1p(F.coalesce(F.col("rl_n"), F.lit(1)) - 1), "relationship", "other customers in the same relationship")
add("dep_sib_d6", chg("_lo", -6, -4), "relationship", "change in the relationship's other balances vs 4–6 months ago")
DEP_FEATS = list(DEP)
X = X.drop("_bl", "_nn", "_nl", "_lo").persist(DISK)
AMP = am.select("acct_full_acct_id", "t", "cust_pwr_id", "live", "acct_bal").persist(DISK)   # for the §7 checks

# ── summary ──────────────────────────────────────────────────────────
g = X.agg(F.count("*").alias("rows"), F.countDistinct("cust_pwr_id").alias("customers"),
          F.sum("live").alias("live_rows"), F.sum("msh").alias("msh_rows")).first().asDict()
lab = topd(X.groupBy("cust_pwr_id").agg(F.first("exit_type").alias("exit_type"), F.first("exit_t").alias("exit_t"),
                                        F.first("a_t").alias("a_t"), F.first("b_t").alias("b_t"))
            .groupBy("exit_type").agg(F.count("*").alias("customers"),
                                      F.sum(F.col("exit_t").isNotNull().cast("int")).alias("also_full_exit_or_provisional"))
            .orderBy("exit_type"))
# is a full exit absorbing?  P(client comes back | ≥ CONFIRM_M observed non-live months)
nl = X.filter((F.col("live") == 0) & (F.col("t") > F.col("first_t")))
nl = nl.withColumn("_g", F.col("t") - F.row_number().over(Window.partitionBy("cust_pwr_id").orderBy("t")))
runs = nl.groupBy("cust_pwr_id", "_g").agg(F.count("*").alias("len"), F.min("t").alias("s"), F.first("last_live_t").alias("llt"))
rr = runs.filter(F.col("len") >= CONFIRM_M).agg(F.count("*").alias("n"), F.sum((F.col("s") < F.col("llt")).cast("int")).alias("back")).first()
R.update(dep_end=END_M, customers=g["customers"], p_return=(rr["back"] or 0) / max(rr["n"], 1), n_dup_keys=n_dup,
         dep_feats=len(DEP_FEATS), pricing_cols=PRC)
B.kv([("deposit window (full months)", f"{DATE_START[:7]} … {END_M} ({END_T + 1} months)"),
      ("duplicated account × leg × day keys (removed)", n_dup), ("customer-months (complete grid)", g["rows"]),
      ("customers", g["customers"]), ("live customer-months", g["live_rows"]),
      ("live months with money still here", f"{g['msh_rows'] / g['live_rows']:.1%}"),
      (f"P(client returns | ≥ {CONFIRM_M} non-live months)", f"{R['p_return']:.2%}  (runs: {rr['n']:,})"),
      ("deposit features", f"{len(DEP_FEATS)} ({', '.join(sorted(set(v[0] for v in DEP.values())))})")], "2a · Panel")
lab["exit_type"] = lab.exit_type.fillna("no exit in window (or not yet confirmed)")
B.table(lab, "2b · Customers by target event (whole window, before any qualification)", save="m1_2b_events")
B.check("deposit panel", "READY", f"{END_M} is the last full month; exits are absorbing ({R['p_return']:.2%} return)"
        if R["p_return"] <= 0.02 else f"{R['p_return']:.2%} of confirmed exits come back — raise CONFIRM_M")
B.show(t0)

## §3 Payment metrics → payment features

Every numeric column of `pkg_cust_metrics_monthly` except the identity columns becomes three features:

| Feature | Meaning |
|---|---|
| `m` | the level: log1p for dollars and counts (signed for net flows), as stored for shares and ratios |
| `m__d6` | change vs the client's own months −6…−4 (a log ratio for dollars and counts, a difference for shares) |
| `m__d3` | change vs the client's own last 3 months |

A client with several `mdm_id`s under one power id takes the row of its largest entity that month. A month with
no row after the client's first row is **zero activity**: dollars and counts 0, shares empty. Churn-state metrics
(new / reactivated / retained / lost, recurring, retention) are left empty for the first 6 months of 2024, while
their 12-month lookback is too short to mean anything.

In [ ]:
# =====================================================================
# 3 · PAYMENT METRICS → PAYMENT FEATURES                         [OUTPUT BLOCK 3]
# =====================================================================
t0 = time.time(); B = Block("§3 · Payment metrics → payment features")
MET = spark.table(MET_TABLE).filter(F.col("month").between(DATE_START[:7], END_M))
ID_KNOWN = {"mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc",
            "naics2", "naics_status", "is_tm", "pwr_shared", "family_size", "customer_start_dt", "in_neo4j_customer",
            "in_cust_dim", "month", "history_months"}
try:
    DICT = topd(spark.table(DICT_TABLE))
    ID_KNOWN |= set(DICT.loc[DICT.family == "identity", "column"])
    FAMILY = dict(zip(DICT.column, DICT.family)); DEFN = dict(zip(DICT.column, DICT.definition))
except Exception as e:
    FAMILY, DEFN = {}, {}
    B.check("dictionary", "INFO", f"{DICT_TABLE} unreadable ({type(e).__name__}); families default to 'other'")
NUM = ("int", "bigint", "double", "float", "smallint", "tinyint", "long")
MCOLS = [c for c, t in MET.dtypes if c not in ID_KNOWN and (t in NUM or t.startswith("decimal"))]

# ── every month present and populated? ───────────────────────────────
nn = topd(MET.groupBy("month").agg(F.count("*").alias("rows"), *[F.count(c).alias(c) for c in MCOLS]).orderBy("month"))
want = [ym_of(t) for t in range(END_T + 1)]
absent = [m for m in want if m not in set(nn.month)]
pop = (nn[MCOLS] > 0)
usual = pop.mean() >= 0.5                                         # columns populated in most months
CHURN_RE = re.compile(r"^(new_|reactivated_|retained_|lost_|rec_|fi_rec_|retention|fi_retention)")
nt = nn.month.map(t_of).to_numpy()
empty = ~pop.loc[:, usual.values]
lookback = np.array([bool(CHURN_RE.match(c)) for c in empty.columns])
# lookback-based columns are legitimately empty while their history is short (first months of 2024)
bad = empty.loc[:, ~lookback].any(axis=1).to_numpy() & (nt >= 1) | empty.any(axis=1).to_numpy() & (nt >= CHURN_WARM)
stale = nn.loc[bad, "month"].tolist()
B.table(nn[["month", "rows"]].assign(empty_columns=empty.sum(axis=1).values),
        "3a · Metrics by month (empty_columns = columns populated in most months but empty here)", save="m1_3a_months")
if absent or stale:
    B.check("metrics months", "BLOCKER", f"absent {absent}, empty columns in {stale}")
    raise RuntimeError(f"metrics months not ready: absent {absent}, stale {stale}")
B.check("metrics months", "READY", f"all {len(want)} months present, {len(MCOLS)} metric columns populated throughout")

# ── key: deposit cust_pwr_id ↔ metrics customer_pwr_id (format decided from the data) ──
def pk_expr(c, mode):
    s = F.trim(F.col(c).cast("string")); s = F.when(s == "", F.lit(None)).otherwise(s)
    return F.regexp_replace(s, "^0+(?=.)", "") if mode == "strip0" else s
dk = X.select("cust_pwr_id").distinct()
mk = MET.select("customer_pwr_id").distinct()
rates = {m_: dk.select(pk_expr("cust_pwr_id", m_).alias("k")).join(
            mk.select(pk_expr("customer_pwr_id", m_).alias("k")).distinct().withColumn("h", F.lit(1)), "k", "left")
            .agg(F.avg(F.coalesce("h", F.lit(0)).cast("double"))).first()[0] for m_ in ("raw", "strip0")}
KEY = "strip0" if rates["strip0"] > rates["raw"] + 0.01 else "raw"
R["key_match"] = rates[KEY]
if R["key_match"] < 0.3: raise RuntimeError(f"power-id join broken: {rates}")

# ── one row per power id and month: the largest entity ───────────────
flow = sum(F.coalesce(F.col(c).cast("double"), F.lit(0.0)) for c in ["amt_all_in", "amt_all_out"] if c in MCOLS)
ACT = [c for c in ["ntxn_all_out", "ntxn_all_in", "ntxn_self_out", "ntxn_self_in"] if c in MCOLS]
pm = (MET.select(pk_expr("customer_pwr_id", KEY).alias("pwr"), t_col("month").alias("t"), "mdm_id",
                 *[F.col(c).cast("double").alias(c) for c in MCOLS])
         .filter("pwr is not null")
         .withColumn("_f", flow)
         .withColumn("_r", F.row_number().over(Window.partitionBy("pwr", "t").orderBy(F.desc("_f"), "mdm_id")))
         .withColumn("pay_n_mdm", F.count("*").over(Window.partitionBy("pwr", "t")))
         .filter("_r = 1").drop("_r", "_f", "mdm_id")
         .withColumn("_row", F.lit(1))
         .withColumn("_act", (sum(F.coalesce(F.col(c), F.lit(0.0)) for c in ACT) > 0).cast("int") if ACT else F.lit(1)))

def kind(c):
    if re.match(r"^(amt_|net_flow|avg_ticket|selfpay_amt)", c): return "money"
    if re.search(r"(share|hhi|retention|entropy|coverage|ratio)", c): return "bounded"
    return "count"
KIND = {c: kind(c) for c in MCOLS}

Y = X.withColumn("pwr", pk_expr("cust_pwr_id", KEY)).join(pm, ["pwr", "t"], "left")
wp = Window.partitionBy("cust_pwr_id").orderBy("t")
Y = (Y.withColumn("pay_first_t", F.min(F.when(F.col("_row") == 1, F.col("t"))).over(
            wp.rowsBetween(Window.unboundedPreceding, 0)))                    # first row known by month t
      .withColumn("pay_seen", F.col("pay_first_t").isNotNull().cast("int"))
      .withColumn("pay_silent", ((F.col("pay_seen") == 1) & F.col("_row").isNull()).cast("int"))
      .withColumn("_last_act", F.max(F.when(F.col("_act") == 1, F.col("t"))).over(wp.rowsBetween(Window.unboundedPreceding, 0)))
      .withColumn("pay_months_since_act", F.col("t") - F.col("_last_act"))
      .withColumn("pay_n_mdm", F.when(F.col("pay_seen") == 1, F.coalesce("pay_n_mdm", F.lit(0)))))
# size anchor and raw activity kept before the transforms (silent month = 0 once the client has been seen)
_z = lambda c: F.coalesce(F.col(c), F.lit(0.0)) if c in MCOLS else F.lit(0.0)
Y = (Y.withColumn("pay_log_flow", F.when(F.col("pay_seen") == 1, F.log1p(F.greatest(_z("amt_all_in") + _z("amt_all_out"), F.lit(0.0)))))
      .withColumn("raw_ntx", F.when(F.col("pay_seen") == 1, _z("ntxn_all_in") + _z("ntxn_all_out")))
      .withColumn("raw_selfpay_out", F.when(F.col("pay_seen") == 1, _z("selfpay_amt_out"))))
lv = []
for c in MCOLS:
    v = F.col(c)
    if KIND[c] == "money": e = F.signum(v) * F.log1p(F.abs(v))
    elif KIND[c] == "count": e = F.log1p(F.greatest(v, F.lit(0.0)))
    else: e = v
    if KIND[c] != "bounded": e = F.when(F.col("_row").isNull() & (F.col("pay_seen") == 1), F.lit(0.0)).otherwise(e)
    if CHURN_RE.match(c): e = F.when(F.col("t") < CHURN_WARM, F.lit(None)).otherwise(e)
    lv.append(e.alias(c))
Y = Y.select(*[c for c in Y.columns if c not in MCOLS], *lv)
wr = lambda a, b: wp.rangeBetween(a, b)
ch = []
for c in MCOLS:
    for nm, (a, b) in (("d6", (-6, -4)), ("d3", (-3, -1))):
        ch.append(F.when(F.count(c).over(wr(a, b)) >= 2, F.col(c) - F.avg(c).over(wr(a, b))).alias(f"{c}__{nm}"))
FEAT = Y.select("*", *ch).persist(DISK)
PAY_FEATS = [f for c in MCOLS for f in (c, f"{c}__d6", f"{c}__d3")]
STATUS_FEATS = ["pay_seen", "pay_silent", "pay_months_since_act"]

cov = FEAT.filter("live = 1").agg(F.count("*").alias("n"), F.avg("pay_seen").alias("seen"), F.avg("pay_silent").alias("silent"),
                                  F.sum(F.when(F.col("pay_seen") == 1, F.col("bal"))).alias("bal_seen"),
                                  F.sum("bal").alias("bal_all")).first()
R.update(pay_seen=cov["seen"], pay_seen_usd=cov["bal_seen"] / cov["bal_all"], pay_silent=cov["silent"],
         n_metrics=len(MCOLS), pay_feats=len(PAY_FEATS), key_mode=KEY)
fam_tab = pd.Series({c: FAMILY.get(c, "other") for c in MCOLS}).value_counts().rename_axis("family").reset_index(name="metrics")
B.kv([("power-id key match (deposit customers found in the metrics)", f"{R['key_match']:.1%} (mode {KEY})"),
      ("live customer-months with payment history", f"{cov['seen']:.1%} of months, {R['pay_seen_usd']:.1%} of balance"),
      ("…of which silent (no metrics row that month)", f"{cov['silent']:.1%} of live months"),
      ("metric columns → features", f"{len(MCOLS)} → {len(PAY_FEATS)} (+ {len(STATUS_FEATS)} activity-status features)")],
     "3b · Payment features")
B.table(fam_tab, "3c · Metric columns by dictionary family")
B.show(t0)

## §4 Model frame: money-still-here rows, peer gaps, de-duplicated features

**Rows:** live, observed ≥ 3 months, normal balance ≥ $10k, current balance ≥ half of normal, money not yet
gone — exactly the clients the list can call. **Label:** the money leaves (full or balance exit) in the next 6
months. **Size peers:** the same month's clients in the same decile of normal balance; each payment level becomes
its gap to the peers' median. **Size anchors:** log balance, log normal balance, log payment flow.

In [ ]:
# =====================================================================
# 4 · MODEL FRAME, TRAINING SAMPLE, FEATURE SETS                  [OUTPUT BLOCK 4]
# =====================================================================
t0 = time.time(); B = Block("§4 · Model frame and feature sets")
LAST_LABEL_T = min(END_T - CONFIRM_M + 1, END_T - BAL_EXIT_HOLD + 1) - H
MF = (FEAT.filter((F.col("live") == 1) & (F.col("hist_t") >= MIN_HIST) & F.col("bal_norm").isNotNull()
                  & (F.col("msh") == 1) & (F.col("ab_t").isNull() | (F.col("t") < F.col("ab_t"))))
          .withColumn("y", F.coalesce(((F.col("ab_t") > F.col("t")) & (F.col("ab_t") <= F.col("t") + H)).cast("int"), F.lit(0)))
          .withColumn("labelled", (F.col("t") <= LAST_LABEL_T).cast("int"))
          .withColumn("dep_log_norm", F.log1p("bal_norm"))
          .withColumn("size_dec", F.least(F.floor(F.percent_rank().over(Window.partitionBy("t").orderBy("bal_norm")) * 10),
                                          F.lit(9)).cast("int")))
wpeer = Window.partitionBy("t", "size_dec")
MF = MF.select("*", *[(F.col(c) - F.expr(f"percentile_approx(`{c}`, 0.5)").over(wpeer)).alias(f"{c}__peer") for c in MCOLS]).persist(DISK)
DEP["dep_log_norm"] = ("size anchor", "normal balance, log"); DEP["dep_log_bal"] = ("size anchor", "balance, log")
ANCHORS = ["dep_log_bal", "dep_log_norm", "pay_log_flow"]
FEATS0 = (ANCHORS + [f for f in DEP_FEATS if f not in ANCHORS] + [f"{c}__peer" for c in MCOLS]
          + [f"{c}__{k}" for c in MCOLS for k in ("d6", "d3")] + STATUS_FEATS)
META = ["cust_pwr_id", "t", "y", "msh", "bal", "bal_norm", "capped", "ab_t", "exit_type", "size_dec"]
keep = F.expr(f"pmod(hash(cust_pwr_id, t, {SEED}), 10000)") < int(NEG_RATE * 10000)
TR = topd(MF.filter("labelled = 1").filter((F.col("y") == 1) | keep)
            .withColumn("w", F.when(F.col("y") == 1, F.lit(1.0)).otherwise(F.lit(1.0 / NEG_RATE)))
            .select(*META, "w", *[F.col(f).cast("float").alias(f) for f in FEATS0]))
TR[FEATS0] = TR[FEATS0].astype("float32")

# ── drop empty / constant, then near-duplicates (keep the better-covered, earlier-listed one) ──
miss0 = TR[FEATS0].isna().mean()
FEATS = [f for f in FEATS0 if miss0[f] < 0.995 and TR[f].nunique(dropna=True) > 1]
S_ = TR[FEATS].sample(min(len(TR), 100_000), random_state=SEED)
S_ = S_.fillna(S_.median()).to_numpy(np.float64)
CM_ = np.nan_to_num(np.corrcoef(S_, rowvar=False))
order = sorted(range(len(FEATS)), key=lambda j: (miss0[FEATS[j]], j))
kept, dupes = [], []
for j in order:
    hit = [i for i in kept if abs(CM_[i, j]) > CORR_MAX]
    if hit: dupes.append({"dropped": FEATS[j], "kept": FEATS[hit[0]], "r": float(CM_[hit[0], j])})
    else: kept.append(j)
FEATS = [FEATS[j] for j in sorted(kept)]
DUP = pd.DataFrame(dupes); DUP.to_csv(OUT / "m2_4_dropped_duplicates.csv", index=False)

def fam_of(f):
    if f in ANCHORS: return "size anchor"
    if f in DEP: return "deposit · " + DEP[f][0]
    if f in STATUS_FEATS: return "payment · activity status"
    if f.startswith("miss·"): return "missing share"
    base = f.split("__")[0]
    return "payment · " + FAMILY.get(base, "other")
def form_of(f):
    if f in ANCHORS: return "size anchor"
    if f.endswith("__peer"): return "gap to size peers"
    if f.endswith("__d6") or f in ("dep_d6", "dep_live_d6", "dep_sib_d6") or f.endswith("_d6"): return "change vs 4–6 months ago"
    if f.endswith("__d3") or f == "dep_d3": return "change vs last 3 months"
    if f in LATE_FEATS: return "drain in progress"
    return "other"
def label(f):
    if f in DEP: return DEP[f][1]
    if f.startswith("miss·"): return f"share of {f[5:]} features not observed"
    st = {"pay_seen": "has payment history", "pay_silent": "no payments this month (silent)",
          "pay_months_since_act": "months since the last payment", "pay_log_flow": "payment flow in + out, log"}
    if f in st: return st[f]
    base, _, tr = f.partition("__")
    d = DEFN.get(base, base.replace("_", " "))
    return d + {"d6": " — change vs 4–6 months ago", "d3": " — change vs the last 3 months",
                "peer": " — gap to size peers", "": ""}[tr]
FAM = {f: fam_of(f) for f in FEATS}
FAM_SETS = sorted(set(v for v in FAM.values() if v.startswith("payment")))
FAM_IDX = {s_: [j for j, f in enumerate(FEATS) if FAM[f] == s_] for s_ in FAM_SETS}
COLS = FEATS + [f"miss·{s_}" for s_ in FAM_SETS]
LABEL = {c: label(c) for c in COLS}; FAMOF = {c: (FAM.get(c) or "missing share") for c in COLS}
FS = pd.DataFrame([{"feature": f, "set": FAM[f], "form": form_of(f), "coverage": 1 - miss0[f]} for f in FEATS])
FST = FS.groupby(["set"]).agg(features=("feature", "size"), median_coverage=("coverage", "median")).reset_index()
FFORM = FS.groupby("form").agg(features=("feature", "size")).reset_index()
FS.assign(label=FS.feature.map(label)).to_csv(OUT / "m2_4_feature_list.csv", index=False)
by_t = topd(MF.groupBy("t").agg(F.count("*").alias("money_still_here"), F.sum("y").alias("leaving_6m"),
                                F.sum(F.when(F.col("y") == 1, F.col("capped"))).alias("leaving_capped"),
                                F.sum("capped").alias("capped")).orderBy("t"))
by_t["month"] = by_t.t.map(ym_of); by_t["rate_6m"] = by_t.leaving_6m / by_t.money_still_here
by_t["labelled"] = by_t.t <= LAST_LABEL_T
R.update(last_label=ym_of(LAST_LABEL_T), train_rows=len(TR), train_pos=int(TR.y.sum()), n_feats=len(FEATS),
         n_dupes=len(DUP), feature_sets=FST, forms=FFORM,
         base_rate=float(by_t.loc[by_t.labelled, "leaving_6m"].sum() / by_t.loc[by_t.labelled, "money_still_here"].sum()))
B.kv([("labelled months", f"… {ym_of(LAST_LABEL_T)} (outcome fully observed)"),
      ("money-still-here client-months, labelled", int(by_t.loc[by_t.labelled, "money_still_here"].sum())),
      ("leave within 6 months (base rate)", f"{R['base_rate']:.2%}"),
      ("training sample", f"{len(TR):,} rows · {R['train_pos']:,} leaving · stayers at {NEG_RATE:.0%}, weight {1 / NEG_RATE:.1f}"),
      ("features", f"{len(FEATS0)} built → {len(FEATS)} kept ({len(DUP)} near-duplicates dropped) + {len(FAM_SETS)} missing shares")],
     "4a · Model frame")
B.table(by_t[["month", "money_still_here", "leaving_6m", "rate_6m", "capped", "leaving_capped", "labelled"]],
        "4b · Money-still-here client-months by month", pct=("rate_6m",), money=("capped", "leaving_capped"), save="m2_4b_by_month")
B.table(FST, "4c · Feature sets", pct=("median_coverage",), save="m2_4c_feature_sets")
B.table(FFORM, "4c · Feature forms")
B.show(t0)

## §5 Rolling monthly test folds — simple model and tree model

For each test month T both models are fitted on months whose outcome was known at T (origins ≤ T − 6). The last
two of those months are held out as validation: on the first fold they choose the ridge penalty and the tree
depth; on every fold they stop the trees early. Both models then score every money-still-here client in T.

In [ ]:
# =====================================================================
# 5 · ROLLING TEST FOLDS — simple (ridge logistic) and tree model   [OUTPUT BLOCK 5]
# =====================================================================
t0 = time.time(); B = Block("§5 · Rolling monthly test folds")
TEST_T = list(range(t_of(TEST_FIRST), LAST_LABEL_T + 1))
if len(TEST_T) < 3: raise RuntimeError(f"only {len(TEST_T)} test months")
FI = [FEATS.index(f) for f in FEATS]

def prep_fit(Xtr):
    s = Xtr[:min(len(Xtr), 200_000)]
    lo, hi = np.nanquantile(s, CLIP_Q[0], axis=0), np.nanquantile(s, CLIP_Q[1], axis=0)
    xc = np.clip(Xtr, lo, hi)
    mu = np.nanmean(xc, axis=0); sd = np.nanstd(xc, axis=0); sd[~(sd > 1e-9)] = 1.0
    mf = np.stack([np.isnan(Xtr[:, FAM_IDX[s_]]).mean(axis=1) for s_ in FAM_SETS], axis=1) if FAM_SETS else np.zeros((len(Xtr), 0))
    return dict(lo=np.nan_to_num(lo), hi=np.nan_to_num(hi), mu=np.nan_to_num(mu), sd=sd, mm=mf.mean(axis=0), ms=mf.std(axis=0) + 1e-9)

def transform(Xr, P):
    z = np.nan_to_num((np.clip(Xr, P["lo"], P["hi"]) - P["mu"]) / P["sd"], nan=0.0)
    mf = np.stack([np.isnan(Xr[:, FAM_IDX[s_]]).mean(axis=1) for s_ in FAM_SETS], axis=1) if FAM_SETS else np.zeros((len(Xr), 0))
    return np.hstack([z, (mf - P["mm"]) / P["ms"]]).astype(np.float32)

def logit_newton(Z, y, w, l2, max_it=60, tol=1e-6):
    """Weighted ridge logistic by DAMPED Newton; intercept unpenalised; chunked."""
    n, k = Z.shape; w = w / w.mean(); CH = 50_000
    pbar = np.average(y, weights=w); b = np.zeros(k + 1); b[0] = np.log(pbar / (1 - pbar))
    def eta(b): return b[0] + np.concatenate([Z[i:i + CH].astype(np.float64) @ b[1:] for i in range(0, n, CH)])
    def obj(e, b): return float(np.sum(w * (np.logaddexp(0, e) - y * e)) + 0.5 * l2 * b[1:] @ b[1:])
    e = eta(b); f = obj(e, b); conv = False
    for it in range(1, max_it + 1):
        p = 1 / (1 + np.exp(-np.clip(e, -35, 35))); r = w * (p - y); v = w * p * (1 - p)
        g = np.empty(k + 1); g[0] = r.sum(); g[1:] = l2 * b[1:]
        Hs = np.zeros((k + 1, k + 1)); Hs[0, 0] = v.sum()
        for i in range(0, n, CH):
            Zc = Z[i:i + CH].astype(np.float64); rc, vc = r[i:i + CH], v[i:i + CH]
            g[1:] += Zc.T @ rc; Hs[1:, 0] += Zc.T @ vc; Hs[1:, 1:] += (Zc * vc[:, None]).T @ Zc
        Hs[0, 1:] = Hs[1:, 0]; Hs[1:, 1:] += l2 * np.eye(k)
        step = np.linalg.solve(Hs, g); s_ = 1.0
        while True:
            bn = b - s_ * step; en = eta(bn); fn = obj(en, bn)
            if fn <= f or s_ < 1e-8: break
            s_ *= 0.5
        b, e, f_old, f = bn, en, f, fn
        if np.max(np.abs(g)) / n < tol or abs(f_old - f) < 1e-10 * max(1.0, abs(f)): conv = True; break
    return b, dict(converged=conv, iters=it)

def lin_score(b, Z): return 1 / (1 + np.exp(-np.clip(b[0] + Z @ b[1:], -35, 35)))

# ── tree engine wrappers: fit(X, y, w, Xv, yv, wv, depth) → model; predict; contributions ──
def gbm_fit(X, y, w, Xv, yv, wv, depth):
    if GBM == "xgboost":
        import xgboost as xgb
        pr = dict(objective="binary:logistic", eval_metric="logloss", eta=GBM_ETA, max_depth=depth, subsample=0.8,
                  colsample_bytree=0.6, min_child_weight=20, reg_lambda=1.0, tree_method="hist", nthread=os.cpu_count(), seed=SEED)
        dt = xgb.DMatrix(X, label=y, weight=w, missing=np.nan); dv = xgb.DMatrix(Xv, label=yv, weight=wv, missing=np.nan)
        m = xgb.train(pr, dt, num_boost_round=GBM_MAX_TREES, evals=[(dv, "val")], early_stopping_rounds=GBM_EARLY, verbose_eval=False)
        return dict(kind="xgboost", m=m, n=int(m.best_iteration) + 1)
    if GBM == "lightgbm":
        import lightgbm as lgb
        pr = dict(objective="binary", learning_rate=GBM_ETA, num_leaves=min(2 ** depth - 1, 255), max_depth=depth,
                  min_data_in_leaf=200, feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbose=-1, seed=SEED, num_threads=os.cpu_count())
        dt = lgb.Dataset(X, y, weight=w); dv = lgb.Dataset(Xv, yv, weight=wv, reference=dt)
        try:    m = lgb.train(pr, dt, GBM_MAX_TREES, valid_sets=[dv], callbacks=[lgb.early_stopping(GBM_EARLY, verbose=False)])
        except (AttributeError, TypeError): m = lgb.train(pr, dt, GBM_MAX_TREES, valid_sets=[dv], early_stopping_rounds=GBM_EARLY, verbose_eval=False)
        return dict(kind="lightgbm", m=m, n=int(m.best_iteration or GBM_MAX_TREES))
    from sklearn.ensemble import HistGradientBoostingClassifier
    m = HistGradientBoostingClassifier(learning_rate=GBM_ETA, max_depth=depth, max_iter=GBM_MAX_TREES, early_stopping=True,
                                       validation_fraction=0.1, n_iter_no_change=GBM_EARLY // 2, l2_regularization=1.0, random_state=SEED)
    m.fit(X, y, sample_weight=w)
    return dict(kind="sklearn", m=m, n=int(m.n_iter_))

def gbm_predict(M, X, contrib=False):
    if M["kind"] == "xgboost":
        import xgboost as xgb
        d = xgb.DMatrix(X, missing=np.nan)
        try:    p = M["m"].predict(d, iteration_range=(0, M["n"]), pred_contribs=contrib)
        except TypeError: p = M["m"].predict(d, ntree_limit=M["n"], pred_contribs=contrib)
        return p[:, :-1] if contrib else p
    if M["kind"] == "lightgbm":
        p = M["m"].predict(X, num_iteration=M["n"], pred_contrib=contrib)
        return p[:, :-1] if contrib else p
    return None if contrib else M["m"].predict_proba(X)[:, 1]

Xall = TR[FEATS].to_numpy(np.float32)
yall, wall, tall = TR.y.to_numpy(float), TR.w.to_numpy(float), TR.t.to_numpy()
TR = TR[META + ["w"]]
L2, DEPTH, TUNE = None, None, []
SCORES, FOLDS, SAMPLE = [], [], []
ABS_L, ABS_G, N_SC = np.zeros(len(COLS)), np.zeros(len(FEATS)), 0
rng = np.random.default_rng(SEED)
for T in TEST_T:
    tf = time.time()
    m = tall <= T - H; mv = m & (tall >= T - H - 1); mf = m & ~mv
    if mv.sum() == 0 or yall[mv].sum() == 0: mf, mv = m, m
    # ── simple model ─────────────────────────────────────────────────
    if L2 is None:
        Pv = prep_fit(Xall[mf]); Zf = transform(Xall[mf], Pv); Zv = transform(Xall[mv], Pv)
        for l2 in L2_GRID:
            bb, _ = logit_newton(Zf, yall[mf], wall[mf], l2)
            TUNE.append({"model": "simple", "setting": f"L2 = {l2:g}", "val_auc": auc(yall[mv], lin_score(bb, Zv))})
        L2 = max((r for r in TUNE if r["model"] == "simple"), key=lambda r: r["val_auc"])["setting"]
        L2 = float(L2.split("= ")[1]); del Zf, Zv
    P = prep_fit(Xall[m]); Z = transform(Xall[m], P)
    b, info = logit_newton(Z, yall[m], wall[m], L2); del Z
    # ── tree model ───────────────────────────────────────────────────
    G = None
    if GBM:
        if DEPTH is None:
            for d_ in GBM_GRID:
                Gd = gbm_fit(Xall[mf], yall[mf], wall[mf], Xall[mv], yall[mv], wall[mv], d_)
                TUNE.append({"model": f"tree ({GBM})", "setting": f"depth {d_}, {Gd['n']} trees",
                             "val_auc": auc(yall[mv], gbm_predict(Gd, Xall[mv]))})
            best = max((r for r in TUNE if r["model"].startswith("tree")), key=lambda r: r["val_auc"])
            DEPTH = int(best["setting"].split(",")[0].split()[1])
        G = gbm_fit(Xall[mf], yall[mf], wall[mf], Xall[mv], yall[mv], wall[mv], DEPTH)
    # ── score the test month ─────────────────────────────────────────
    te = topd(MF.filter(F.col("t") == T).select(*META, *[F.col(f).cast("float").alias(f) for f in FEATS]))
    Xt = te[FEATS].to_numpy(np.float32); Zt = transform(Xt, P)
    Cl = Zt * b[1:].astype(np.float32); te["p_lin"] = lin_score(b, Zt)
    ABS_L += np.abs(Cl).sum(axis=0); N_SC += len(te)
    Cg = None
    if G:
        te["p_gbm"] = gbm_predict(G, Xt); Cg = gbm_predict(G, Xt, contrib=True)
        if Cg is not None: ABS_G += np.abs(Cg).sum(axis=0)
    else:
        te["p_gbm"] = np.nan
    RC, RN = (Cg, FEATS) if Cg is not None else (Cl, COLS)          # reason codes: tree SHAP when available
    top = np.argsort(-RC, axis=1)[:, :3]
    for k_ in range(3):
        te[f"r{k_ + 1}"] = [RN[j] for j in top[:, k_]]
    te["tm"] = T
    si = rng.random(len(te)) < min(1.0, 15_000 / max(len(te), 1))
    SAMPLE.append(pd.DataFrame(Xt[si], columns=FEATS).assign(y=te.y.values[si]))
    SCORES.append(te.drop(columns=FEATS))
    FOLDS.append({"test_month": ym_of(T), "train_rows": int(m.sum()), "train_leaving": int(yall[m].sum()),
                  "simple_converged": info["converged"], "trees": (G or {}).get("n"), "test_rows": len(te),
                  "test_leaving": int(te.y.sum()), "auc_simple": auc(te.y, te.p_lin), "auc_tree": auc(te.y, te.p_gbm) if G else np.nan,
                  "mean_p_simple": float(te.p_lin.mean()), "mean_p_tree": float(te.p_gbm.mean()) if G else np.nan,
                  "actual_rate": float(te.y.mean()), "minutes": (time.time() - tf) / 60})
    MODEL = dict(b=b, P=P, G=G)
    print(f"  {ym_of(T)}: {len(te):,} scored · AUC simple {FOLDS[-1]['auc_simple']:.3f} · tree {FOLDS[-1]['auc_tree']:.3f} · "
          f"{FOLDS[-1]['minutes']:.1f} min")
SC = pd.concat(SCORES, ignore_index=True); SMP = pd.concat(SAMPLE, ignore_index=True); FO = pd.DataFrame(FOLDS)
SC["capped"] = SC.capped.fillna(0.0).clip(lower=0)
PRIMARY = "p_gbm" if GBM else "p_lin"
SC["p"] = SC[PRIMARY]
R.update(test_months=[ym_of(t) for t in TEST_T], n_test=len(TEST_T), folds=FO, tune=pd.DataFrame(TUNE), L2=L2, depth=DEPTH,
         engine=GBM, primary="tree model" if GBM else "simple model",
         auc_lin=auc(SC.y, SC.p_lin), auc_gbm=auc(SC.y, SC.p_gbm) if GBM else np.nan)
B.table(pd.DataFrame(TUNE), "5a · Settings chosen on the first fold's validation months", save="m2_5a_tuning")
B.table(FO, "5b · One fold per test month", pct=("mean_p_simple", "mean_p_tree", "actual_rate"), save="m2_5b_folds")
B.kv([("pooled AUC, simple model", R["auc_lin"]), ("pooled AUC, tree model", R["auc_gbm"]),
      ("tree − simple", R["auc_gbm"] - R["auc_lin"] if GBM else np.nan),
      ("AUC range across months (tree)", f"{FO.auc_tree.min():.3f} – {FO.auc_tree.max():.3f}" if GBM else "—"),
      ("primary model for the list, reasons and examples", R["primary"])], "5c · Discrimination")
if not GBM: B.check("tree model", "FIX", "no tree library in new_default: pip install xgboost (or lightgbm) and rerun")
B.show(t0)

## §6 Predictive power

**In the model:** each feature's share of the average absolute contribution on the test months — exact TreeSHAP
for the tree model, coefficient × standardised value for the simple model. **Alone:** how well the feature ranks
leavers by itself (AUC folded so 0.5 = nothing). Features are also grouped by *form* (size anchor, gap to size
peers, change, drain in progress), which answers whether the model now runs on change rather than size.

In [ ]:
# =====================================================================
# 6 · PREDICTIVE POWER                                              [OUTPUT BLOCK 6]
# =====================================================================
t0 = time.time(); B = Block("§6 · Predictive power")
alone = {}
for f in FEATS:
    a = auc(SMP.y, SMP[f].to_numpy(float))
    alone[f] = (max(a, 1 - a) if pd.notna(a) else np.nan, "higher" if (pd.notna(a) and a >= 0.5) else "lower")
use_g = GBM is not None and ABS_G.sum() > 0
shr = (ABS_G / ABS_G.sum()) if use_g else (ABS_L[:len(FEATS)] / ABS_L.sum())
lin_shr = ABS_L / ABS_L.sum()
IMP = pd.DataFrame([{"feature": f, "label": LABEL[f], "set": FAM[f], "form": form_of(f), "share": shr[j],
                     "share_simple": lin_shr[j], "simple_sign": "risk up when high" if MODEL["b"][1 + j] > 0 else "risk up when low",
                     "alone_auc": alone[f][0], "leavers_sit": alone[f][1]} for j, f in enumerate(FEATS)])
IMP = IMP.sort_values("share", ascending=False).reset_index(drop=True)
IMP.to_csv(OUT / "m2_6_importance_all.csv", index=False)
FAMI = IMP.groupby("set").agg(share=("share", "sum"), share_simple=("share_simple", "sum"), features=("feature", "size"),
                              best_alone=("alone_auc", "max")).sort_values("share", ascending=False).reset_index()
if not use_g: FAMI.loc[len(FAMI)] = ["missing share", float(lin_shr[len(FEATS):].sum()), float(lin_shr[len(FEATS):].sum()), len(FAM_SETS), np.nan]
FORM = IMP.groupby("form").agg(share=("share", "sum"), share_simple=("share_simple", "sum"), features=("feature", "size")) \
          .sort_values("share", ascending=False).reset_index()
R.update(imp=IMP, fam_imp=FAMI, form_imp=FORM, imp_basis="tree SHAP" if use_g else "simple model contributions",
         miss_share_simple=float(lin_shr[len(FEATS):].sum()))
B.table(IMP.head(25), f"6a · Top 25 features ({R['imp_basis']}; share_simple = the simple model's share)",
        pct=("share", "share_simple"), save="m2_6a_top")
B.table(FAMI, "6b · Feature sets", pct=("share", "share_simple"), save="m2_6b_sets")
B.table(FORM, "6c · Feature forms: size, peer gap, change, drain in progress", pct=("share", "share_simple"), save="m2_6c_forms")
B.kv([("missing-share columns in the simple model", f"{R['miss_share_simple']:.1%} of contribution (model 1: 24.5% from 89 flags)")],
     "6d · Missingness")
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
t20 = IMP.head(20)[::-1]
ax[0].barh([l[:60] for l in t20.label], t20.share, color="#1f4e79"); ax[0].set_title(f"Top 20 features ({R['imp_basis']})")
f12 = FAMI.head(12)[::-1]
ax[1].barh(f12.set, f12.share, color="#c55a11"); ax[1].set_title("Feature sets")
for a in ax: a.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m2_6_importance")
B.show(t0)

## §7 Checking the largest departures

A handful of very large clients carry most of the dollars, so each of the largest departures is checked in the
data before its dollars are counted. For every client: did the balance **come back** (≥ 50% of normal after the
exit month)? Did any of its accounts get **re-linked** to another customer id? Did **other customers in the same
relationship** gain what this one lost (money moving inside PNC)? Does the client **keep transacting** here
afterwards? Were payments to **its own name at other banks** visible around the exit? Verdicts:

| Verdict | Rule | Counted as a loss? |
|---|---|---|
| came back | balance returned to ≥ 50% of normal | no |
| moved within PNC | an account re-linked, or siblings absorbed ≥ 50% of the drop | no |
| still transacting | payment activity after the exit ≥ 50% of before | yes, flagged |
| departed | none of the above | yes |

In [ ]:
# =====================================================================
# 7 · CHECKING THE LARGEST DEPARTURES                              [OUTPUT BLOCK 7]
# =====================================================================
t0 = time.time(); B = Block("§7 · Checking the largest departures")
ex = topd(FEAT.filter(F.col("ab_t").isNotNull() & F.col("ab_t").between(MIN_HIST + 1, LAST_LABEL_T + H))
              .groupBy("cust_pwr_id").agg(F.first("ab_t").alias("ab_t"), F.first("exit_type").alias("exit_type"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 1, F.col("bal_norm"))).alias("norm"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 3, F.col("msh"))).alias("msh_m3"),
                                          F.max(F.when(F.col("t") == F.col("ab_t") - 3, F.col("capped"))).alias("capped_m3")))
ex = ex.dropna(subset=["norm"])
AUD_CACHE = {}
def audit(ids):
    """Evidence and verdict for each departing client id (cached)."""
    new = [c for c in dict.fromkeys(ids) if c not in AUD_CACHE]
    if new:
        P_ = topd(FEAT.filter(F.col("cust_pwr_id").isin(new)).select("cust_pwr_id", "t", "bal", "bal_norm", "live", "ab_t",
                                                                     "exit_type", "first_t", "rltn_pwr_id", "raw_ntx", "raw_selfpay_out"))
        accs = [r[0] for r in AMP.filter(F.col("cust_pwr_id").isin(new)).select("acct_full_acct_id").distinct().collect()]
        RL = topd(AMP.filter(F.col("acct_full_acct_id").isin(accs or ["__none__"]))
                     .select("acct_full_acct_id", "t", "cust_pwr_id", "live"))
        rls = P_.dropna(subset=["rltn_pwr_id"]).groupby("cust_pwr_id").rltn_pwr_id.last()
        SIB = topd(FEAT.filter(F.col("rltn_pwr_id").isin(list(set(rls.values)) or ["__none__"]))
                       .groupBy("rltn_pwr_id", "t", "cust_pwr_id").agg(F.sum("bal").alias("bal"))) if len(rls) else pd.DataFrame(
                       columns=["rltn_pwr_id", "t", "cust_pwr_id", "bal"])
        for c in new:
            p = P_[P_.cust_pwr_id == c].sort_values("t").set_index("t")
            if not len(p) or p.ab_t.isna().all(): AUD_CACHE[c] = {"verdict": "no exit"}; continue
            e = int(p.ab_t.dropna().iloc[0]); norm = p.bal_norm.get(e - 1, np.nan)
            pre = p.bal.loc[e - 3:e - 1].mean(); post = p.bal.loc[e:e + 2].mean()
            back = p.bal.loc[e + 1:].max() / norm if norm and len(p.bal.loc[e + 1:]) else np.nan
            r_ = RL[RL.acct_full_acct_id.isin(RL.loc[RL.cust_pwr_id == c, "acct_full_acct_id"])]
            relinked = int(r_[(r_.t >= e - 1) & (r_.cust_pwr_id != c)].acct_full_acct_id.nunique())
            absorb = np.nan
            if c in rls.index:
                sb = SIB[(SIB.rltn_pwr_id == rls[c]) & (SIB.cust_pwr_id != c)].groupby("t").bal.sum()
                if len(sb) and pre - post > 0:
                    absorb = (sb.loc[e:e + 2].mean() - sb.loc[e - 3:e - 1].mean()) / (pre - post)
            act_b = p.raw_ntx.loc[e - 6:e - 4].mean(); act_a = p.raw_ntx.loc[e:e + 2].mean()
            act = act_a / act_b if act_b and act_b > 0 else np.nan
            selfp = p.raw_selfpay_out.loc[e - 3:e].sum() / (pre - post) if pre - post > 0 else np.nan
            if pd.notna(back) and back >= 0.5: v = "came back"
            elif relinked > 0 or (pd.notna(absorb) and absorb >= 0.5): v = "moved within PNC"
            elif pd.notna(act) and act >= 0.5: v = "still transacting"
            else: v = "departed"
            AUD_CACHE[c] = {"client": cid(c), "exit_type": p.exit_type.dropna().iloc[0] if p.exit_type.notna().any() else None,
                            "exit_month": ym_of(e), "normal": norm, "months_observed_before": int(e - p.first_t.iloc[0]),
                            "balance_after_vs_normal": back, "accounts_relinked": relinked, "siblings_absorbed": absorb,
                            "activity_after_vs_before": act, "own_name_out_vs_drop": selfp,
                            "live_accounts_at_end": int(r_[(r_.t == END_T) & (r_.cust_pwr_id == c)].live.sum()),
                            "verdict": v}
    return pd.DataFrame([{"cust_pwr_id": c, **AUD_CACHE[c]} for c in dict.fromkeys(ids)])

pool_all = (SC[SC.y == 1].sort_values("tm").groupby("cust_pwr_id")
              .agg(ab_t=("ab_t", "first"), exit_type=("exit_type", "first"), capped=("capped", "max"),
                   bal_norm=("bal_norm", "first"), tm=("tm", "first")).reset_index())
top_ids = list(ex.sort_values("norm", ascending=False).cust_pwr_id.head(N_AUDIT)) + \
          list(pool_all.sort_values("capped", ascending=False).cust_pwr_id.head(N_AUDIT))
AUD = audit(top_ids)
AUD = AUD.merge(pool_all[["cust_pwr_id", "capped"]].rename(columns={"capped": "test_pool_capped"}), on="cust_pwr_id", how="left")
AUD.to_csv(OUT / "m2_7_audit_FULL_IDS.csv", index=False)
EXCL = set(AUD.loc[AUD.verdict.isin(["came back", "moved within PNC"]), "cust_pwr_id"])
VC = AUD.groupby("verdict").agg(clients=("client", "size"), normal=("normal", "sum")).reset_index()
R.update(audit=AUD.drop(columns="cust_pwr_id"), audit_counts=VC, n_not_clean=int((AUD.verdict != "departed").sum()),
         n_audited=len(AUD), excl_ids=EXCL)
B.table(AUD.drop(columns="cust_pwr_id"), f"7a · The largest departures ({len(AUD)} clients: top {N_AUDIT} of the window, top {N_AUDIT} of the test months)",
        pct=("balance_after_vs_normal", "siblings_absorbed", "activity_after_vs_before", "own_name_out_vs_drop"),
        money=("normal", "test_pool_capped"))
B.table(VC, "7b · Verdicts", money=("normal",))
B.check("largest departures", "INFO", f"{len(EXCL)} of {len(AUD)} are not counted as losses (came back or moved within PNC); "
        "full ids in m2_7_audit_FULL_IDS.csv for an RM check")
B.show(t0)

## §8 How much is at risk; accuracy and savings against calls

The list each month: money-still-here clients ranked by `p × normal balance^α`, top K called. A departing client
is credited once, at its first alert, with its capped balance (the lesser of current and normal). Dollars are
annualised over the test months. Savings = dollars reached × an assumed save rate; there is no call cost.
Headline dollars are also shown **without the largest 1 and 5 departing clients** and **without the clients §7
found were not real losses**. Simple and tree model are compared on the same rows with a paired bootstrap over
departing clients.

In [ ]:
# =====================================================================
# 8 · AT RISK · ACCURACY VS CALLS · SAVINGS · MODEL COMPARISON      [OUTPUT BLOCK 8]
# =====================================================================
t0 = time.time(); B = Block("§8 · At risk, accuracy, savings")
span = (LAST_LABEL_T + H) - (MIN_HIST + 1) + 1
ex["band"] = pd.cut(ex.norm, [b[0] for b in BANDS] + [BANDS[-1][1]], labels=[b[2] for b in BANDS], right=False)
exv = ex[~ex.cust_pwr_id.isin(EXCL)]
AR = exv.groupby(["band", "exit_type"], observed=False).agg(clients=("norm", "size"), normal_balance=("norm", "sum")).reset_index()
AR["per_year"] = AR.normal_balance * 12 / span
R.update(at_risk_year=exv.norm.sum() * 12 / span, exits_per_year=len(exv) * 12 / span, at_risk_tab=AR,
         msh_m3=float(exv.msh_m3.mean()), capped_m3_year=float(exv.capped_m3.fillna(0).sum() * 12 / span),
         bal_exit_share=float(exv.loc[exv.exit_type == "balance exit", "norm"].sum() / exv.norm.sum()),
         top_client_share=float(exv.norm.max() / exv.norm.sum()), excl_normal=float(ex.loc[ex.cust_pwr_id.isin(EXCL), "norm"].sum() * 12 / span))
B.kv([("clients whose money left (per year, checked exclusions removed)", f"{R['exits_per_year']:,.0f}"),
      ("their normal balance (per year)", f"${R['at_risk_year']:,.0f}"),
      ("…of which balance exits (accounts left open)", f"{R['bal_exit_share']:.1%}"),
      ("…still here 3 months before (clients) / capped dollars per year", f"{R['msh_m3']:.1%} / ${R['capped_m3_year']:,.0f}"),
      ("largest single client's share", f"{R['top_client_share']:.1%}"),
      ("removed by §7 (came back / moved within PNC), per year", f"${R['excl_normal']:,.0f}")], "8a · How much is at risk")
B.table(AR, "8a · By size band and exit type (annualised)", money=("normal_balance", "per_year"), save="m2_8a_at_risk")

Q = SC.copy(); Q["norm"] = Q.bal_norm.clip(lower=1.0)
yr = 12 / R["n_test"]
MODELS = [("simple", "p_lin")] + ([("tree", "p_gbm")] if GBM else [])
first_alert, per_month = {}, []
for mn, pc in MODELS:
    for a in ALPHAS:
        Q[f"rank_{mn}_{a}"] = Q.assign(_s=Q[pc] * Q.norm ** a).groupby("tm")["_s"].rank(ascending=False, method="first")
        for K in CALLS:
            on = Q[Q[f"rank_{mn}_{a}"] <= K]
            for T, g in Q.groupby("tm"):
                o = on[on["tm"] == T]; lv = g[g.y == 1]
                per_month.append({"model": mn, "alpha": a, "calls": K, "month": ym_of(T), "precision": o.y.mean(),
                                  "recall_clients": o.y.sum() / max(len(lv), 1),
                                  "recall_dollars": o.loc[o.y == 1, "capped"].sum() / max(lv.capped.sum(), 1)})
            first_alert[(mn, a, K)] = on[on.y == 1].sort_values("tm").drop_duplicates("cust_pwr_id")
PM_ = pd.DataFrame(per_month)
pool = pool_all.copy(); POOL = pool.capped.sum()
top_pool = list(pool.sort_values("capped", ascending=False).cust_pwr_id)
def reached(fa, drop=()):
    f = fa[~fa.cust_pwr_id.isin(drop)]; return f.capped.sum() * yr, len(f)
rows = []
for (mn, a, K), fa in first_alert.items():
    d_, n_ = reached(fa)
    rows.append({"model": mn, "alpha": a, "calls": K, "clients_reached": n_, "share_of_departing": n_ / max(len(pool), 1),
                 "dollars_reached_year": d_, "share_of_dollars": d_ / yr / max(POOL, 1),
                 **{f"save_{int(s * 100)}pct_year": d_ * s for s in SAVE_RATES}})
VAL = pd.DataFrame(rows)
ACC = (PM_.groupby(["model", "alpha", "calls"]).agg(precision_mean=("precision", "mean"), precision_min=("precision", "min"),
                                                    precision_max=("precision", "max"), recall_clients=("recall_clients", "mean"),
                                                    recall_dollars=("recall_dollars", "mean")).reset_index())
MAIN = "tree" if GBM else "simple"
fa0 = first_alert[(MAIN, A_MAIN, K_MAIN)]
variants = [("all departing clients", ()), *[(f"without the largest {k}", tuple(top_pool[:k])) for k in TOP_EXCL],
            ("without clients §7 found were not losses", tuple(EXCL))]
HV = []
for nm, drop in variants:
    d_, n_ = reached(fa0, drop); pl = pool[~pool.cust_pwr_id.isin(drop)].capped.sum() * yr
    HV.append({"dollars": nm, "clients_reached": n_, "reached_per_year": d_, "departing_per_year": pl, "share": d_ / max(pl, 1)})
HV = pd.DataFrame(HV)
# paired bootstrap over departing clients: tree − simple at the main operating point
val_of = lambda mn: pool.cust_pwr_id.map(first_alert[(mn, A_MAIN, K_MAIN)].set_index("cust_pwr_id").capped).fillna(0.0).to_numpy()
cs = val_of("simple"); ct = val_of("tree") if GBM else cs
bs_t, bs_d = [], []
for _ in range(2000):
    i = rng.integers(0, len(pool), len(pool)); bs_t.append(ct[i].sum() * yr); bs_d.append((ct[i] - cs[i]).sum() * yr)
_pm = PM_[(PM_.model == MAIN) & (PM_.alpha == A_MAIN) & (PM_.calls == K_MAIN)]
R.update(pool_year=POOL * yr, n_departing=len(pool), val=VAL, acc=ACC, per_month=PM_, headline=HV,
         main_reached=float(HV.reached_per_year.iloc[0]), main_ci=(float(np.percentile(bs_t, 2.5)), float(np.percentile(bs_t, 97.5))),
         main_share=float(HV.share.iloc[0]), diff=float((ct - cs).sum() * yr),
         diff_ci=(float(np.percentile(bs_d, 2.5)), float(np.percentile(bs_d, 97.5))),
         prec_ratio=float(_pm.precision.max() / _pm.precision.min()) if _pm.precision.min() > 0 else np.inf,
         drop_top5=1 - float(HV.reached_per_year.iloc[2] / HV.reached_per_year.iloc[0]) if len(HV) > 2 and HV.reached_per_year.iloc[0] else np.nan)
CMP = VAL[(VAL.alpha == A_MAIN)].pivot(index="calls", columns="model", values="dollars_reached_year").reset_index()
CMP = CMP.merge(ACC[ACC.alpha == A_MAIN].pivot(index="calls", columns="model", values="precision_mean").add_prefix("precision_").reset_index(), on="calls")
B.kv([(f"departing clients with money still here, test months", f"{len(pool):,} · ${POOL * yr:,.0f} capped per year"),
      (f"reached at {K_MAIN:,} calls, α = {A_MAIN} ({MAIN} model)",
       f"{len(fa0):,} clients · ${R['main_reached']:,.0f}/yr (95% {R['main_ci'][0]:,.0f} – {R['main_ci'][1]:,.0f})"),
      ("tree − simple, dollars reached per year", f"${R['diff']:,.0f} (95% {R['diff_ci'][0]:,.0f} – {R['diff_ci'][1]:,.0f})" if GBM else "—"),
      ("best ÷ worst month precision", f"{R['prec_ratio']:.2f}×")], "8b · Headline")
B.table(HV, f"8b · Headline dollars at {K_MAIN:,} calls, α = {A_MAIN}, with and without the largest clients",
        pct=("share",), money=("reached_per_year", "departing_per_year"), save="m2_8b_headline")
B.table(CMP, f"8c · Simple vs tree model at α = {A_MAIN}: dollars reached per year and precision",
        pct=tuple(c for c in CMP.columns if c.startswith("precision_")),
        money=tuple(c for c in CMP.columns if c in ("simple", "tree")), save="m2_8c_compare")
B.table(ACC[ACC.model == MAIN], f"8d · Accuracy against calls ({MAIN} model; mean over test months, min/max = spread)",
        pct=("precision_mean", "precision_min", "precision_max", "recall_clients", "recall_dollars"), save="m2_8d_accuracy")
B.table(VAL[VAL.model == MAIN].drop(columns="model"), f"8e · Dollars reached and potential savings per year ({MAIN} model, no call cost)",
        pct=("share_of_departing", "share_of_dollars"),
        money=tuple(["dollars_reached_year"] + [f"save_{int(s * 100)}pct_year" for s in SAVE_RATES]), save="m2_8e_value")
BYM = (PM_[(PM_.calls == K_MAIN) & (PM_.alpha == A_MAIN)].pivot(index="month", columns="model", values="precision")
          .add_prefix("precision_").reset_index().merge(FO[["test_month", "auc_simple", "auc_tree"]], left_on="month",
                                                        right_on="test_month").drop(columns="test_month"))
B.table(BYM, f"8f · By test month at {K_MAIN:,} calls, α = {A_MAIN}", pct=tuple(c for c in BYM.columns if c.startswith("precision_")),
        save="m2_8f_by_month")
SC["bin"] = pd.qcut(SC.p.rank(method="first"), 10, labels=False)
CAL = SC.groupby("bin").agg(mean_predicted=("p", "mean"), actual=("y", "mean"), rows=("y", "size")).reset_index()
CAL["ratio"] = CAL.actual / CAL.mean_predicted
R.update(cal=CAL, bym=BYM, cmp=CMP)
B.table(CAL, f"8g · Calibration ({MAIN} model), predicted vs actual by decile", pct=("mean_predicted", "actual"))

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a in ALPHAS:
    d = ACC[(ACC.model == MAIN) & (ACC.alpha == a)]; v = VAL[(VAL.model == MAIN) & (VAL.alpha == a)]
    ax[0].plot(d.calls, d.precision_mean, marker="o", label=f"α={a}"); ax[1].plot(d.calls, d.recall_dollars, marker="o", label=f"α={a}")
    ax[2].plot(v.calls, v.dollars_reached_year / 1e6, marker="o", label=f"α={a}")
for mn, _ in MODELS:
    v = VAL[(VAL.model == mn) & (VAL.alpha == A_MAIN)]
    ax[2].plot(v.calls, v.dollars_reached_year / 1e6, ls=":", color="grey" if mn == "simple" else "black", label=f"{mn}, α={A_MAIN}")
for a_, t_ in zip(ax, ["Precision", "Share of departing dollars reached (per month)", "Dollars reached per year ($m)"]):
    a_.set_title(t_); a_.set_xlabel("calls per month"); a_.legend(fontsize=8)
ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m2_8_accuracy_vs_calls")
fig, ax = plt.subplots(1, len(SAVE_RATES), figsize=(4 * len(SAVE_RATES), 3.6))
for i, s in enumerate(SAVE_RATES):
    for a in ALPHAS:
        v = VAL[(VAL.model == MAIN) & (VAL.alpha == a)]; ax[i].plot(v.calls, v[f"save_{int(s * 100)}pct_year"] / 1e6, marker="o", label=f"α={a}")
    ax[i].set_title(f"Savings/yr at {s:.0%} save rate ($m)"); ax[i].set_xlabel("calls per month"); ax[i].legend(fontsize=8)
savefig(fig, "m2_8_savings_vs_calls")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for mn, _ in MODELS:
    ax[0].plot(BYM.month, BYM[f"precision_{mn}"], marker="o", label=mn)
    ax[1].plot(FO.test_month, FO[f"auc_{mn}"], marker="o", label=mn)
ax[0].set_title(f"Precision by test month ({K_MAIN:,} calls, α={A_MAIN})"); ax[1].set_title("AUC by test month")
for a_ in ax: a_.legend(); a_.tick_params(axis="x", rotation=90)
ax[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
savefig(fig, "m2_8_by_month")
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.plot(CAL.mean_predicted, CAL.actual, marker="o"); mx = max(CAL.mean_predicted.max(), CAL.actual.max())
ax.plot([0, mx], [0, mx], color="grey", lw=0.8); ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("Calibration by decile")
savefig(fig, "m2_8_calibration")
B.show(t0)

## §9 When signals fire

**Event time** (describes behaviour; uses the exit date): money-still-here months of leavers aligned on the month
the money left, against stayers aligned on a pseudo-exit month drawn by a fixed hash. Only **change** features are
traced, so a client's size cannot pass for a signal, and every comparison is made **within the same size band**:
thresholds (stayers' 10th/90th percentile) are set per band and the band lifts are averaged with the leavers' band
mix. Onset = the earliest month from which the lift stays ≥ 1.5 through month −1. Drains already in progress are
listed separately as **late warnings**. **Model view:** each feature's contribution to leavers' scores minus
stayers', month by month; the total gap shows whether the score rises toward the exit. **Queue time:** how many
months before the money left a client first reached the list.

In [ ]:
# =====================================================================
# 9 · WHEN SIGNALS FIRE                                              [OUTPUT BLOCK 9]
# =====================================================================
t0 = time.time(); B = Block("§9 · When signals fire")
CHG_FORMS = ("change vs 4–6 months ago", "change vs last 3 months")
CHG = [f for f in IMP.feature if form_of(f) in CHG_FORMS or f in ("dep_bal_vs_norm", "dep_drawdown")]
TIMING = CHG[:N_TIMING] + [f for f in LATE_FEATS if f in FEATS]
span_pe = (LAST_LABEL_T + H) - 12 + 1
ev_base = (MF.withColumn("e", F.when(F.col("ab_t").isNotNull(), F.col("ab_t"))
                              .when(F.col("last_live_t") == END_T, F.lit(12) + F.expr(f"pmod(hash(cust_pwr_id), {span_pe})")))
             .withColumn("grp", F.when(F.col("ab_t").isNotNull(), F.lit("leaver")).otherwise(F.lit("stayer")))
             .filter(F.col("e").isNotNull() & ((F.col("grp") == "stayer") | (F.col("e") <= LAST_LABEL_T + H)))
             .withColumn("rel", F.col("t") - F.col("e")).filter(F.col("rel").between(-EVENT_PRE, -1)))
_cnt = {r["grp"]: r["n"] for r in ev_base.groupBy("grp").agg(F.countDistinct("cust_pwr_id").alias("n")).collect()}
def sub(n_per_group, salt):
    fr = {g: int(1e6 * min(1.0, n_per_group / max(_cnt.get(g, 1), 1))) for g in ("leaver", "stayer")}
    return ev_base.filter(F.expr(f"pmod(hash(cust_pwr_id, {salt}), 1000000)") <
                          F.when(F.col("grp") == "leaver", F.lit(fr["leaver"])).otherwise(F.lit(fr["stayer"])))
bandx = F.when(F.col("bal_norm") < 1e5, 0).when(F.col("bal_norm") < 1e6, 1).when(F.col("bal_norm") < 1e7, 2).otherwise(3)
EV = topd(sub(MAX_EVENT_CUST, SEED + 1).select("cust_pwr_id", "grp", "rel", bandx.alias("band"),
                                               *[F.col(f).cast("double").alias(f) for f in dict.fromkeys(TIMING)]))
lv_rows, st_rows = EV[EV.grp == "leaver"], EV[EV.grp == "stayer"]
rels = list(range(-EVENT_PRE, 0))
wts = lv_rows.groupby(["rel", "band"]).size().unstack(fill_value=0).reindex(rels, fill_value=0)
tim, LIFT = [], {}
for f in dict.fromkeys(TIMING):
    Lb = {"falls": [], "rises": []}
    for bnd in sorted(EV.band.unique()):
        s_ = st_rows[st_rows.band == bnd][f].dropna()
        if len(s_) < 50: continue
        q10, q90 = s_.quantile(0.10), s_.quantile(0.90)
        e_ = EV[EV.band == bnd]; x = e_[f]
        for side, fire in (("falls", (x < q10) & x.notna()), ("rises", (x > q90) & x.notna())):
            r_ = fire.groupby([e_.grp, e_.rel]).mean()
            if "leaver" in r_.index.get_level_values(0) and "stayer" in r_.index.get_level_values(0):
                lift = (r_["leaver"] / r_["stayer"].replace(0, np.nan)).reindex(rels)
                w_ = (wts[bnd] if bnd in wts.columns else pd.Series(0, index=rels)).astype(float)
                Lb[side].append((lift.fillna(0) * w_ * lift.notna(), w_ * lift.notna()))
    comb = {k: (sum(n for n, _ in v) / sum(d for _, d in v).replace(0, np.nan)) if v else pd.Series(np.nan, index=rels)
            for k, v in Lb.items()}
    side = "falls" if np.nanmean(comb["falls"].loc[-3:-1]) >= np.nanmean(comb["rises"].loc[-3:-1]) else "rises"
    Lf = comb[side]; on = None
    for r_ in range(-1, -EVENT_PRE - 1, -1):
        if pd.notna(Lf.get(r_)) and Lf[r_] >= LIFT_BAR: on = r_
        else: break
    LIFT[f] = Lf
    tim.append({"feature": f, "label": LABEL[f], "set": FAM[f], "kind": "late warning" if f in LATE_FEATS else "early signal",
                "leavers": side, "onset_month": on, **{f"lift_{-k}m": Lf.get(k) for k in (-9, -6, -3, -1)},
                "coverage": float(lv_rows[f].notna().mean())})
TIM = pd.DataFrame(tim).sort_values(["kind", "onset_month", "lift_1m"], ascending=[True, True, False], na_position="last").reset_index(drop=True)
R.update(tim=TIM, n_pay_onset6=int(((TIM.onset_month <= -6) & TIM.set.str.startswith("payment")).sum()),
         n_ev_leavers=int(lv_rows.cust_pwr_id.nunique()), n_ev_stayers=int(st_rows.cust_pwr_id.nunique()))
B.table(TIM, "9a · Event time, size-matched: when each change signal fires (lift = leavers ÷ stayers of the same size band)",
        pct=("coverage",), save="m2_9a_onset")
fig, ax = plt.subplots(figsize=(12, 0.32 * len(TIM) + 1.5))
M_ = np.array([LIFT[f].values for f in TIM.feature], dtype=float)
im = ax.imshow(np.clip(M_, 0, 5), aspect="auto", cmap="Reds", vmin=1, vmax=5)
ax.set_yticks(range(len(TIM))); ax.set_yticklabels([f"{l[:62]} ({d})" for l, d in zip(TIM.label, TIM.leavers)], fontsize=8)
ax.set_xticks(range(len(rels))); ax.set_xticklabels(rels); ax.set_xlabel("months before the money left")
ax.set_title("Lift over size-matched stayers (1 = no difference, capped at 5)"); fig.colorbar(im, ax=ax, shrink=0.6)
savefig(fig, "m2_9a_onset_heatmap")

# ── model view: contributions, leavers minus stayers, by month ───────
EVS = topd(sub(N_SHAP_CUST, SEED + 2).select("grp", "rel", *[F.col(f).cast("float").alias(f) for f in FEATS]))
Xe = EVS[FEATS].to_numpy(np.float32)
Ce = gbm_predict(MODEL["G"], Xe, contrib=True) if MODEL["G"] else None
if Ce is None: Ce = (transform(Xe, MODEL["P"]) * MODEL["b"][1:].astype(np.float32))[:, :len(FEATS)]
CEdf = pd.DataFrame(Ce, columns=FEATS).assign(grp=EVS.grp.values, rel=EVS.rel.values)
mg = CEdf.groupby(["grp", "rel"]).mean()
gap = (mg.loc["leaver"] - mg.loc["stayer"]).reindex(rels)
TOTAL = gap.sum(axis=1)
top15 = list(IMP.feature.head(15))
MVT = gap[top15].T; MVT.columns = [str(c) for c in rels]
MVT.insert(0, "feature", [LABEL[f][:70] for f in top15]); MVT = MVT.reset_index(drop=True)
MVT.loc[len(MVT)] = ["ALL FEATURES (total log-odds gap)"] + list(TOTAL.values)
g6 = TOTAL.get(-6, np.nan); g1 = TOTAL.get(-1, np.nan)
R.update(model_view=MVT, gap6=float(g6), gap1=float(g1), gap_ratio=float(g6 / g1) if g1 else np.nan,
         lead6=[LABEL[f][:60] for f in gap.loc[-6].sort_values(ascending=False).head(5).index],
         lead1=[LABEL[f][:60] for f in gap.loc[-1].sort_values(ascending=False).head(5).index])
B.table(MVT, "9b · Model view: contribution to leavers' score minus stayers' (log-odds) by month before the money left",
        save="m2_9b_model_view")
B.kv([("total gap 6 months out ÷ 1 month out", f"{R['gap_ratio']:.2f} (model 1 was ≈ 0.8: flat)"),
      ("largest drivers 6 months out", " · ".join(R["lead6"])), ("largest drivers 1 month out", " · ".join(R["lead1"]))],
     "9b · Early vs late drivers")
fig, ax = plt.subplots(figsize=(8, 3.6)); ax.plot(rels, TOTAL.values, marker="o", color="#c00000")
ax.set_title("Leavers' score minus stayers' (log-odds), all features"); ax.set_xlabel("months before the money left")
savefig(fig, "m2_9b_total_gap")

# ── queue time ───────────────────────────────────────────────────────
fq = fa0.copy(); fq["lead"] = (fq.ab_t - fq["tm"]).astype(int)
fq["reason_set"] = fq.r1.map(lambda r: FAMOF.get(r, "?")); fq["reason_kind"] = fq.r1.map(lambda r: "late warning" if r in LATE_FEATS else "early signal")
LEAD = fq.groupby("lead").agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index()
LEAD["share_clients"] = LEAD.clients / LEAD.clients.sum(); LEAD["share_dollars"] = LEAD.capped / LEAD.capped.sum()
RS = fq.groupby(["reason_kind", "reason_set"]).agg(clients=("cust_pwr_id", "size"), capped=("capped", "sum")).reset_index() \
       .sort_values("clients", ascending=False)
RS["share"] = RS.clients / RS.clients.sum()
R.update(lead=LEAD, reason_sets=RS, lead3=float((fq.lead >= 3).mean()) if len(fq) else np.nan,
         lead3_usd=float(fq.loc[fq.lead >= 3, "capped"].sum() / max(fq.capped.sum(), 1)),
         late_share=float((fq.reason_kind == "late warning").mean()) if len(fq) else np.nan)
B.table(LEAD, f"9c · Queue time: months from the first alert to the money leaving ({K_MAIN:,} calls, α = {A_MAIN})",
        pct=("share_clients", "share_dollars"), money=("capped",), save="m2_9c_lead")
B.table(RS, "9c · Top reason at the first alert", pct=("share",), money=("capped",))

# ── leavers vs stayers on change features ────────────────────────────
BEHAV = [f for f in ["dep_bal_vs_norm", "dep_d6", "ntxn_all_out__d6", "ntxn_all_in__d6", "amt_all_out__d6", "amt_all_in__d6",
                     "n_fi_ach_out__d6", "n_fi_out__d6", "ncpty_all_out__d6", "ncpty_all_in__d6", "active_days_out__d6",
                     "active_days_in__d6"] if f in MF.columns][:12]
EB = topd(sub(MAX_EVENT_CUST, SEED + 1).select("grp", "rel", *[F.col(f).cast("double").alias(f) for f in BEHAV]))
BV = EB.groupby(["grp", "rel"])[BEHAV].mean()
fig, axs = plt.subplots(int(np.ceil(len(BEHAV) / 4)), 4, figsize=(16, 3.1 * int(np.ceil(len(BEHAV) / 4))))
for a_, f in zip(np.ravel(axs), BEHAV):
    for g_, col in (("leaver", "#c00000"), ("stayer", "#1f4e79")):
        a_.plot(rels, BV.loc[g_, f].reindex(rels), marker="o", ms=3, color=col, label=g_)
    a_.axhline(0, color="grey", lw=0.6); a_.set_title(label(f)[:50], fontsize=9); a_.tick_params(labelsize=8)
for a_ in np.ravel(axs)[len(BEHAV):]: a_.axis("off")
np.ravel(axs)[0].legend(fontsize=8); fig.suptitle("Mean change (log ratio) by month before the money left — money-still-here months")
savefig(fig, "m2_9d_leavers_vs_stayers")
LS = pd.DataFrame([{"feature": label(f), "month": k, "leavers": BV.loc["leaver", f].get(k), "stayers": BV.loc["stayer", f].get(k)}
                   for f in BEHAV for k in (-12, -6, -3, -1)]).pivot(index="feature", columns="month", values=["leavers", "stayers"])
LS.columns = [f"{a} {b}" for a, b in LS.columns]; LS = LS.reset_index()
R.update(ls=LS)
B.table(LS, "9d · Leavers vs stayers: mean change 12, 6, 3 and 1 months before", save="m2_9d_leavers_vs_stayers")
B.kv([("leavers / stayers traced", f"{R['n_ev_leavers']:,} / {R['n_ev_stayers']:,}"),
      ("payment change features firing 6+ months out (size-matched)", R["n_pay_onset6"]),
      ("first alerted 3+ months before the money left", f"{R['lead3']:.1%} of clients, {R['lead3_usd']:.1%} of dollars"),
      ("first alerts whose top reason is a drain already in progress", f"{R['late_share']:.1%}")], "9e · Summary")
B.show(t0)

## §10 Example clients for RM managers

Chosen **by rule** from the test months, at 1,000 calls and α = 0.5 on the primary model, so nobody hand-picked a flattering case:
an early catch, a case where payments moved before the balance, a client who drained without closing, a late
catch, the most confident false alarm, and the largest miss. Each shows the balance against normal, the months
on the list, the reasons the model gave, the event-time signals that fired, and the window an RM had to act.
Every departing example must pass the §7 check (verdict *departed*). Ids are one-way hashes; full ids go only to
`m2_10_examples_FULL_IDS.csv`. Review each with its RM before showing it more widely.

In [ ]:
# =====================================================================
# 10 · EXAMPLE CLIENTS FOR RM MANAGERS                             [OUTPUT BLOCK 10]
# =====================================================================
t0 = time.time(); B = Block("§10 · Example clients")
def play(feat):
    f = feat.replace("miss·", "")
    if re.search(r"(n_fi|^fi_|_fi_|selfpay|top_fi|^share_|n_rails)", f):
        return ("Competitive threat: money appears to be moving to another bank. Compare pricing, earnings credit "
                "and services with the likely competitor; offer a retention package.")
    if f.startswith("dep_"):
        return ("Balance drawdown: deposits are falling ahead of the payments. Ask about cash plans; offer sweep, "
                "money-market or liquidity options to keep the operating balance here.")
    return ("Business activity is shrinking: fewer payments and trading partners. Check the client's business; "
            "offer credit, working-capital or payment-product support.")

rk = f"rank_{MAIN}_{A_MAIN}"
Qx = Q[["cust_pwr_id", "tm", rk, "p", "bal", "bal_norm", "capped", "ab_t", "exit_type", "y", "r1", "r2", "r3"]]
# first month each departing client was on the list before its money left (any month, not only the last six)
lst = (Qx[(Qx[rk] <= K_MAIN) & (Qx.tm < Qx.ab_t)].sort_values("tm").drop_duplicates("cust_pwr_id")
         [["cust_pwr_id", "tm", "bal", "bal_norm", "r1"]]
         .rename(columns={"tm": "T_alert", "bal": "bal_at_alert", "bal_norm": "norm_at_alert"}))
dp = pool[["cust_pwr_id", "ab_t", "exit_type", "capped", "bal_norm"]].merge(lst, on="cust_pwr_id", how="left")
dp["lead"] = dp.ab_t - dp.T_alert
dp["pct_at_alert"] = dp.bal_at_alert / dp.norm_at_alert
hit = dp[dp.T_alert.notna()]
chosen, EXS = set(), []
def pick(df, kind, sort="bal_norm"):
    d = df[~df.cust_pwr_id.isin(chosen)].sort_values(sort, ascending=False).head(12)
    if not len(d): return
    v = audit(list(d.cust_pwr_id)).set_index("cust_pwr_id").verdict
    ok = [c for c in d.cust_pwr_id if v.get(c) == "departed"]
    if ok: chosen.add(ok[0]); EXS.append((kind, ok[0]))
pick(hit[hit.lead >= 4], "Early catch")
pick(hit[(hit.pct_at_alert >= 0.9) & hit.r1.map(lambda r: isinstance(r, str) and not r.startswith("dep_"))],
     "Payments moved before the balance")
pick(hit[hit.exit_type == "balance exit"], "Drained without closing")
pick(hit[hit.lead <= 2], "Late catch")
never = set(SC.loc[SC.ab_t.isna(), "cust_pwr_id"]) - set(SC.loc[SC.y == 1, "cust_pwr_id"])
fal = Qx[(Qx[rk] <= K_MAIN) & Qx.cust_pwr_id.isin(never)].sort_values("p", ascending=False)
if len(fal): chosen.add(fal.cust_pwr_id.iloc[0]); EXS.append(("False alarm (most confident wrong call)", fal.cust_pwr_id.iloc[0]))
pick(dp[dp.T_alert.isna()], "Missed", sort="capped")

ids = [c for _, c in EXS]
TIMF = [f for f in dict.fromkeys(TIMING) if f in FEAT.columns]
PAN = topd(FEAT.filter(F.col("cust_pwr_id").isin(ids))
               .select("cust_pwr_id", "t", "bal", "bal_norm", "live", "ab_t", *[F.col(f).cast("double").alias(f) for f in TIMF]))
tim_on = TIM.dropna(subset=["onset_month"])
tim_on = tim_on[tim_on.feature.isin(TIMF)]
Q10 = st_rows[TIMF].quantile(0.10); Q90 = st_rows[TIMF].quantile(0.90)
EXOUT, full_ids = [], []
for n, (kind, c) in enumerate(EXS, 1):
    pan = PAN[PAN.cust_pwr_id == c].sort_values("t"); sc = Qx[Qx.cust_pwr_id == c].sort_values("tm")
    e = pan.ab_t.dropna().iloc[0] if pan.ab_t.notna().any() else None
    lo_t = int(min(sc["tm"].min() if len(sc) else pan.t.max(), (e - 12) if e is not None else 10 ** 6))
    hi_t = int(e + 1) if e is not None else int(sc["tm"].max() + 1 if len(sc) else pan.t.max())
    pan = pan[(pan.t >= max(lo_t, 0)) & (pan.t <= hi_t)]
    ref = float(sc.bal_norm.iloc[0]) if len(sc) else float(pan.bal_norm.dropna().median())
    # event-time signals that fired for this client (same thresholds and directions as §8a)
    fired = []
    for _, s in tim_on.iterrows():
        f = s.feature; th = Q10[f] if s.leavers == "falls" else Q90[f]
        v = pan.set_index("t")[f]
        hitm = v[(v < th) if s.leavers == "falls" else (v > th)].index
        if len(hitm) and e is not None and (hitm < e).any():
            fired.append((int(min(hitm[hitm < e])) - int(e), s.label))
    fired = sorted(fired)[:5]
    alerts = sc[sc[rk] <= K_MAIN]
    rows = []
    for _, r in pan.iterrows():
        s_ = sc[sc["tm"] == r.t]
        rows.append({"month": ym_of(r.t), "vs_exit": (int(r.t - e) if e is not None else None),
                     "balance_pct_normal": r.bal / ref if ref else np.nan,
                     "score": float(s_.p.iloc[0]) if len(s_) else np.nan,
                     "rank": f"{int(s_[rk].iloc[0]):,}" if len(s_) else "—",
                     "on_list": bool(len(s_) and s_[rk].iloc[0] <= K_MAIN),
                     "reasons": " · ".join(LABEL.get(x, x)[:60] for x in s_[["r1", "r2", "r3"]].iloc[0]) if len(s_) else ""})
    TAB = pd.DataFrame(rows)
    first = alerts.iloc[0] if len(alerts) else None
    if kind.startswith("False"):
        text = (f"On the list in {len(alerts)} test month(s) with a score up to {sc.p.max():.1%}, and stayed. "
                f"Reasons given: {TAB.loc[TAB.on_list, 'reasons'].iloc[0] if TAB.on_list.any() else '—'}. "
                f"This is what a wrong call looks like; at {K_MAIN:,} calls about "
                f"{1 - float(ACC[(ACC.model == MAIN) & (ACC.alpha == A_MAIN) & (ACC.calls == K_MAIN)].precision_mean.iloc[0]):.0%} of calls are like it.")
    elif first is None:
        best = sc.sort_values(rk).iloc[0] if len(sc) else None
        text = (f"The money left in {ym_of(e)} ({dp.loc[dp.cust_pwr_id == c, 'exit_type'].iloc[0]}) "
                f"and the client never reached the top {K_MAIN:,}" +
                (f"; best rank {int(best[rk]):,} in {ym_of(best['tm'])}, reasons: {' · '.join(LABEL.get(x, x)[:50] for x in best[['r1','r2','r3']])}."
                 if best is not None else ".") +
                (" Signals that did fire: " + "; ".join(f"{l[:55]} (from {m})" for m, l in fired) + "." if fired else ""))
    else:
        pct = first.bal / first.bal_norm if first.bal_norm else np.nan
        text = (f"First on the list in {ym_of(first['tm'])}, {int(e - first['tm'])} month(s) before the money left "
                f"({dp.loc[dp.cust_pwr_id == c, 'exit_type'].iloc[0]}, {ym_of(e)}). The balance was then {pct:.0%} of its "
                f"${first.bal_norm:,.0f} normal. Reasons: {' · '.join(LABEL.get(x, x)[:60] for x in first[['r1','r2','r3']])}. " +
                ("Signals that fired: " + "; ".join(f"{l[:55]} (from {m})" for m, l in fired) + ". " if fired else "") +
                f"How an RM could have intervened: {play(first.r1)}")
    fig, ax = plt.subplots(figsize=(10, 3.4))
    ax.plot([ym_of(t) for t in pan.t], pan.bal / ref, marker="o", color="#1f4e79", label="balance, % of normal")
    for _, a_ in alerts.iterrows():
        if a_["tm"] in set(pan.t): ax.axvline(ym_of(a_["tm"]), color="#c55a11", alpha=0.35, lw=9)
    if e is not None and e in set(pan.t): ax.axvline(ym_of(e), color="#c00000", ls="--", label="money left")
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0)); ax.tick_params(axis="x", rotation=90)
    ax.set_title(f"Example {n} · {kind} · client {cid(c)} (orange = on the list)"); ax.legend(loc="lower left", fontsize=8)
    savefig(fig, f"m2_10_example_{n}")
    EXOUT.append(dict(n=n, kind=kind, client=cid(c), normal=ref, text=text, table=TAB, chart=f"m2_10_example_{n}"))
    full_ids.append({"example": n, "kind": kind, "display_id": cid(c), "cust_pwr_id": c})
    B.note(f"<b>Example {n} · {kind} · client {cid(c)}</b> — {_html.escape(text)}")
    B.table(TAB, f"Example {n}: month by month", pct=("balance_pct_normal", "score"))
pd.DataFrame(full_ids).to_csv(OUT / "m2_10_examples_FULL_IDS.csv", index=False)
R.update(examples=EXOUT)
if len(EXS) < 6: B.check("examples", "INFO", f"only {len(EXS)} of 6 rules found a client")
B.show(t0)

## §11–12 Prediction scorecard and the HTML report

Same sections as before, plus the model comparison and the check of the largest departures. Section 1 ("What we
found") stays a placeholder with the headline numbers filled in; the findings are written after the results are read.

In [ ]:
# =====================================================================
# 12 · PREDICTIONS · HTML REPORT                                    [OUTPUT BLOCK 12]
# =====================================================================
t0 = time.time(); B = Block("§12 · Predictions and report")
_v = lambda mn, a, K: float(VAL[(VAL.model == mn) & (VAL.alpha == a) & (VAL.calls == K)].dollars_reached_year.iloc[0])
PRED = [
    ("P1", "Tree AUC ≥ simple + 0.02", 0.60, lambda: R["auc_gbm"] - R["auc_lin"] >= 0.02, f"{R['auc_gbm'] - R['auc_lin']:+.3f}"),
    ("P2", "Tree reaches more dollars, 95% paired interval > 0", 0.45, lambda: R["diff_ci"][0] > 0,
     f"${R['diff']:,.0f} ({R['diff_ci'][0]:,.0f} – {R['diff_ci'][1]:,.0f})"),
    ("P3", "Gap at −6 ≤ 60% of gap at −1", 0.55, lambda: R["gap_ratio"] <= 0.60, f"{R['gap_ratio']:.2f}"),
    ("P4", "≥ 3 payment change features fire 6+ months out", 0.45, lambda: R["n_pay_onset6"] >= 3, f"{R['n_pay_onset6']}"),
    ("P5", "≥ 3 of the largest departures are not clean", 0.50, lambda: R["n_not_clean"] >= 3, f"{R['n_not_clean']} of {R['n_audited']}"),
    ("P6", "Without the top 5, dollars reached fall ≥ 30%", 0.50, lambda: R["drop_top5"] >= 0.30, f"{R['drop_top5']:.1%}"),
    ("P7", "Best ÷ worst month precision ≤ 2×", 0.60, lambda: R["prec_ratio"] <= 2, f"{R['prec_ratio']:.2f}×")]
rows = []
for pid, txt, conf, fn, obs in PRED:
    try: hit = bool(fn())
    except Exception: hit = None
    rows.append({"id": pid, "prediction": txt, "confidence": conf, "observed": obs,
                 "result": "HELD" if hit else "FAILED" if hit is False else "n/a",
                 "brier": (conf - float(hit)) ** 2 if hit is not None else np.nan})
PS = pd.DataFrame(rows); R["preds"] = PS
B.table(PS, "12a · Pre-registered predictions", pct=("confidence",), save="m2_12a_predictions")

def img(name):
    return f"<img src='data:image/png;base64,{base64.b64encode(open(CHARTS[name], 'rb').read()).decode()}' style='max-width:100%'>"
def tab(df, pct=(), money=(), rows=60):
    return fmt_df(df.head(rows), pct, money).to_html(index=False, border=0, classes="t", escape=True)
usd = lambda v: f"${v / 1e9:,.2f}bn" if abs(v) >= 1e9 else f"${v / 1e6:,.1f}m" if abs(v) >= 1e6 else f"${v:,.0f}"
HVd = R["headline"]
S = [f"""<h1>Deposit attrition early warning — v13 model 2</h1>
<p class='sub'>Payment Knowledge Graph (PKG) · Treasury Management · Data Science · test months {R['test_months'][0]} … {R['test_months'][-1]} ·
primary model: {R['primary']}{f' ({R["engine"]})' if R['engine'] else ''}</p>"""]
S.append(f"""<section><h2>1 · What we found</h2><div class='box warn'><b>To be written after the results are read.</b>
Headline numbers, filled in automatically:</div>{tab(pd.DataFrame([
 ["clients whose money left, per year (after the §7 check)", f"{R['exits_per_year']:,.0f}"],
 ["their normal balance, per year", usd(R['at_risk_year'])], ["…share leaving through balance exits (accounts left open)", f"{R['bal_exit_share']:.1%}"],
 ["pooled AUC, simple / tree", f"{R['auc_lin']:.3f} / {R['auc_gbm']:.3f}" if R['engine'] else f"{R['auc_lin']:.3f} / —"],
 [f"dollars reached per year, {K_MAIN:,} calls, α = {A_MAIN}", f"{usd(R['main_reached'])} (95% {usd(R['main_ci'][0])} – {usd(R['main_ci'][1])})"],
 ["…without the largest 5 departing clients", usd(float(HVd.reached_per_year.iloc[2])) if len(HVd) > 2 else "—"],
 ["tree − simple, dollars reached per year", f"{usd(R['diff'])} (95% {usd(R['diff_ci'][0])} – {usd(R['diff_ci'][1])})" if R['engine'] else "—"],
 ["first alerted 3+ months before the money left", f"{R['lead3']:.1%} of clients, {R['lead3_usd']:.1%} of dollars"],
 ["leavers' score gap 6 months out ÷ 1 month out", f"{R['gap_ratio']:.2f}"]], columns=["measure", "value"]))}</section>""")
S.append("""<section><h2>2 · The question</h2><p>Which Treasury Management deposit clients will move their money out in the
next six months, early enough for a relationship manager to act; how much of that money a fixed number of calls can
reach; and how much a stronger model adds. Model 2 corrects model 1, which had mostly learned which <i>kind</i> of
client leaves (small, quiet) instead of what <i>changes</i> before a client leaves.</p></section>""")
S.append(f"""<section><h2>3 · Data and preparation</h2>
<h3>Sources</h3>{tab(pd.DataFrame([[DEP_TABLE, "daily balances and status per account × leg"], [MET_TABLE, f"{R['n_metrics']} monthly payment metrics per customer"]], columns=["table", "content"]))}
<h3>Pipeline</h3><ol><li>Legs summed per account-day; balances rolled to customer-day, then month; both status vocabularies mapped to live / closed.</li>
<li>A complete month grid per customer: a month with no rows is non-live with zero balance.</li>
<li>Payment metrics joined on the power id ({R['key_match']:.1%} of deposit customers found); the largest entity per power id and month.</li>
<li>Rows kept: <b>money still here</b> only. Payment levels become gaps to size peers (same month, same decile of normal
balance); changes vs months −6…−4 and vs the last 3 months; three size anchors.</li>
<li>{R['n_dupes']} near-duplicate features dropped (|r| &gt; {CORR_MAX}); one missing-share column per feature set in the simple model;
the tree model reads missing values natively.</li></ol>
<h3>Silent months</h3><p>No payment row after the client's first row = zero activity (dollars and counts 0; shares empty).
{R['pay_seen']:.1%} of live client-months have payment history; {R['pay_silent']:.1%} are silent.</p>
<h3>Checks</h3>{tab(pd.DataFrame(CHECKS))}</section>""")
S.append(f"""<section><h2>4 · Definitions</h2><h3>The target, and why</h3>
<div class='box'><b>The money leaves within the next 6 months</b>: the earlier of a <b>full exit</b> (every account non-live,
confirmed by {CONFIRM_M} observed non-live months) and a <b>balance exit</b> (balance below {BAL_EXIT_FRAC:.0%} of normal for
{BAL_EXIT_HOLD} months while accounts stay open).</div><ul>
<li><b>It is what costs money:</b> {R['bal_exit_share']:.0%} of departing normal balance leaves through balance exits, which a closure-only target never sees.</li>
<li><b>It is earlier:</b> the drain usually precedes the formal closure.</li>
<li><b>Full exits are absorbing:</b> {R['p_return']:.2%} of clients non-live for {CONFIRM_M}+ months came back.</li>
<li><b>Six months</b> leaves time to act; months without six confirmed months ahead ({R['last_label']} onwards) are unlabelled.</li></ul>
<h3>Terms</h3>{tab(pd.DataFrame([
 ["normal balance", f"median balance over up to {NORM_WIN} prior live months (needs {NORM_MIN_OBS})"],
 ["money still here", f"live, normal ≥ ${MSH_FLOOR:,.0f}, current ≥ {MSH_FRAC:.0%} of normal — the only rows trained on, scored and called"],
 ["size peers", "same month, same decile of normal balance"], ["capped credit", "lesser of current and normal balance at the alert"],
 ["α", "score = probability × normal balance^α"], ["dollars reached", "capped balance of departing clients at first alert, counted once, annualised"],
 ["savings", "dollars reached × assumed save rate (1–20%); no call cost"]], columns=["term", "meaning"]))}</section>""")
S.append(f"""<section><h2>5 · Training and test sets</h2><p>Rolling monthly folds: each test month's models see only outcomes known then
(origins ≤ test month − 6); the last two of those months are validation (ridge penalty and tree depth on the first fold;
early stopping on every fold). Stayers are sampled at {NEG_RATE:.0%} and weighted back.</p>
{tab(R['tune'])}{tab(R['folds'], pct=('mean_p_simple', 'mean_p_tree', 'actual_rate'))}</section>""")
S.append(f"""<section><h2>6 · Feature sets</h2><p>{R['n_feats']} features after de-duplication; no identity columns.</p>
{tab(R['feature_sets'], pct=('median_coverage',))}{tab(R['forms'])}</section>""")
S.append(f"""<section><h2>7 · Methods</h2><h3>Simple model</h3><p>Ridge logistic regression (L2 = {R['L2']:g}, chosen on validation) on
standardised, clipped features plus one missing-share per feature set; damped Newton.</p>
<h3>Tree model</h3><p>{('Gradient-boosted trees (' + R['engine'] + f'), depth {R["depth"]}, learning rate {GBM_ETA}, early stopping on the validation months; exact TreeSHAP gives the reason codes.') if R['engine'] else 'Not run: no tree library installed.'}</p>
<h3>Validation</h3><p>Backward-looking features; labels need six confirmed months; each fold trains only on outcomes known before its test month; test months are scored in full.</p>
<h3>From a score to dollars</h3><p>Money-still-here clients ranked by p × normal^α; top K called; each departing client credited once, at its first alert, with its capped balance.</p>
<h3>Uncertainty and concentration</h3><p>Per-month accuracy; bootstrap over departing clients for the dollars and for the tree-vs-simple difference; dollars shown without the largest clients and after the §7 check.</p></section>""")
S.append(f"""<section><h2>8 · Results</h2>
<h3>8.1 How much is at risk</h3><p>{R['exits_per_year']:,.0f} clients a year move their money out, with {usd(R['at_risk_year'])} of normal balance
({R['bal_exit_share']:.0%} through balance exits). The largest single client is {R['top_client_share']:.1%} of it.</p>{tab(R['at_risk_tab'], money=('normal_balance', 'per_year'))}
<h3>8.2 Checking the largest departures</h3>{tab(R['audit_counts'], money=('normal',))}{tab(R['audit'], pct=('balance_after_vs_normal', 'siblings_absorbed', 'activity_after_vs_before', 'own_name_out_vs_drop'), money=('normal', 'test_pool_capped'))}
<h3>8.3 Which features predict</h3>{img('m2_6_importance')}{tab(R['form_imp'], pct=('share', 'share_simple'))}
{tab(R['imp'].head(25), pct=('share', 'share_simple'))}{tab(R['fam_imp'], pct=('share', 'share_simple'))}
<h3>8.4 Simple vs tree model</h3>{tab(R['cmp'], pct=tuple(c for c in R['cmp'].columns if c.startswith('precision_')), money=tuple(c for c in R['cmp'].columns if c in ('simple', 'tree')))}
<h3>8.5 Accuracy against calls</h3>{img('m2_8_accuracy_vs_calls')}{tab(R['acc'], pct=('precision_mean', 'precision_min', 'precision_max', 'recall_clients', 'recall_dollars'))}
<h3>8.6 Accuracy by test month</h3>{img('m2_8_by_month')}{tab(R['bym'], pct=tuple(c for c in R['bym'].columns if c.startswith('precision_')))}
<h3>8.7 Calibration</h3>{img('m2_8_calibration')}{tab(R['cal'], pct=('mean_predicted', 'actual'))}
<h3>8.8 When signals fire</h3><p>Event time describes leavers after the fact; queue time is what an RM experiences.</p>
{img('m2_9a_onset_heatmap')}{tab(R['tim'], pct=('coverage',))}<h4>Model view</h4>{img('m2_9b_total_gap')}{tab(R['model_view'])}
<p>Largest drivers 6 months out: {', '.join(R['lead6'])}. 1 month out: {', '.join(R['lead1'])}.</p>
<h4>Queue time</h4>{tab(R['lead'], pct=('share_clients', 'share_dollars'), money=('capped',))}{tab(R['reason_sets'], pct=('share',), money=('capped',))}
<h3>8.9 Savings against calls</h3>{img('m2_8_savings_vs_calls')}{tab(HVd, pct=('share',), money=('reached_per_year', 'departing_per_year'))}
{tab(R['val'], pct=('share_of_departing', 'share_of_dollars'), money=tuple(['dollars_reached_year'] + [f'save_{int(s * 100)}pct_year' for s in SAVE_RATES]))}
<h3>8.10 Leavers and stayers</h3>{img('m2_9d_leavers_vs_stayers')}{tab(R['ls'])}
<h3>8.11 Examples for RM managers</h3><p>Chosen by rule; every departing example passed the §7 check. Review with the client's RM before wider use.</p>""" +
"".join(f"<h4>Example {e['n']} · {e['kind']} · client {e['client']}</h4><p>{_html.escape(e['text'])}</p>{img(e['chart'])}"
        f"{tab(e['table'], pct=('balance_pct_normal', 'score'))}" for e in R["examples"]) + "</section>")
S.append(f"""<section><h2>9 · Predictions written before the run</h2>{tab(R['preds'], pct=('confidence',))}</section>
<section><h2>10 · Limitations</h2><ul><li><b>The save rate is unmeasured</b>; savings are illustrations until a pilot with a holdout.</li>
<li><b>The §7 check is a rule, not a review</b>; the flagged clients need an RM's confirmation.</li>
<li><b>Identity is current-state:</b> a customer's power id and relationship are today's.</li>
<li><b>Few test months</b> ({R['n_test']}): the data start in 2024 and every label needs six confirmed months.</li>
<li><b>Event-time onsets describe leavers after the fact</b>; they are not detection leads.</li></ul></section>
<section><h2>11 · Next steps</h2><ul><li>RM review of the largest departures and the example clients.</li>
<li>A pilot with a randomised holdout to measure the save rate.</li><li>Tune the tree model further only if the dollars-reached interval says it pays.</li></ul></section>
<section><h2>Appendix · Parameters</h2>{tab(pd.DataFrame([[k, str(v)] for k, v in [("H", H), ("CONFIRM_M", CONFIRM_M), ("NORM_WIN / MIN_OBS", f"{NORM_WIN} / {NORM_MIN_OBS}"),
 ("money still here", f"{MSH_FRAC} × normal, ≥ ${MSH_FLOOR:,}"), ("balance exit", f"< {BAL_EXIT_FRAC} × normal for {BAL_EXIT_HOLD} months"),
 ("NEG_RATE", NEG_RATE), ("L2 chosen", R['L2']), ("tree depth chosen", R['depth']), ("CORR_MAX", CORR_MAX), ("CALLS", CALLS),
 ("ALPHAS", ALPHAS), ("SAVE_RATES", SAVE_RATES), ("LIFT_BAR", LIFT_BAR), ("N_AUDIT", N_AUDIT)]], columns=["parameter", "value"]))}</section>""")
CSS = """body{font-family:Arial,Helvetica,sans-serif;max-width:1180px;margin:24px auto;padding:0 18px;color:#222;line-height:1.45}
h1{font-size:26px;margin-bottom:4px}.sub{color:#666;margin-top:0}h2{border-bottom:2px solid #1f4e79;padding-bottom:4px;margin-top:34px}
h3{color:#1f4e79;margin-top:22px}table.t{border-collapse:collapse;font-size:12px;margin:8px 0;display:block;overflow-x:auto}
table.t th{background:#1f4e79;color:#fff;padding:4px 8px;text-align:left}table.t td{padding:3px 8px;border-bottom:1px solid #e3e3e3}
.box{background:#eef4fa;border-left:4px solid #1f4e79;padding:10px 14px;margin:10px 0}.box.warn{background:#fff4e5;border-color:#e65100}"""
REPORT = OUT / "PKG_Attrition_v13_m2_report.html"
REPORT.write_text(f"<!doctype html><html><head><meta charset='utf-8'><title>PKG attrition v13 · model 2</title><style>{CSS}</style></head>"
                  f"<body>{''.join(S)}<p style='color:#999;font-size:11px'>Internal — PNC Treasury Management, Data Science</p></body></html>",
                  encoding="utf-8")
B.kv([("report", str(REPORT.resolve())), ("size", f"{REPORT.stat().st_size / 1e6:,.1f} MB"), ("CSVs and charts", str(OUT.resolve())),
      ("total wall time", f"{(time.time() - T0) / 60:,.1f} min")], "12b · Written")
for d_ in (X, FEAT, MF, AMP): d_.unpersist()
B.show(t0)